In [204]:
!lscpu


'lscpu' is not recognized as an internal or external command,
operable program or batch file.


# Library

In [2]:
# Import các thư viện cần thiết

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, Flatten, Input
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

import os
# import cv2


import matplotlib.pyplot as plt
import seaborn as sns

# Mining Dataset

# Xử lý outfits

In [61]:
ROOT= r"E:\DoCode\CD2\source\Source\get_hrs_rs\rs\get10k_data\output_10k_sample"
ROOT_PRESS = r"E:\DoCode\CD2\source\Source\get_hrs_rs\rs\get10k_data\output_10k_sample\preprocess"


In [5]:
with open(rf"{ROOT_PRESS}\outfits_10k.csv", 'r') as f:
    lines = f.readlines()

# In các dòng đầu tiên để kiểm tra định dạng
for i in range(5):
    print(lines[i])


id;name;description;group;owner;timeCreated;retailPrice;pricePerWeek;pricePerMonth;outfit_tags;tag_categories

outfit.ffd83466cdb84a0dba02339aa0c72f73;Mariposa Earrings AB-Crystal;The Mariposa Earrings are the perfect statement jewelry. We recommend wearing your hair up to show them off. ;group.4f70fa1707b559c0db341fa53997e52f;o_00037;2019-03-22 11:52:11.000;1600.0;250.0;500.0;['Jewelry', 'Cecilie Melli', 'Statement', 'Metallic', 'Formal'];['Category', 'Brand', 'Occasion', 'Details', 'Occasion']

outfit.ff08dd87bf144defaa332eac0bf5bd4e;The Gisele Dress;This mini satin dress is made in a beautiful olive color, and features a V-neckline and puffed sleeves. The pleated detail in the front gives it an edge. ;group.a65e95a2f1823633a8d0b07b7fff2e7b;o_00740;2020-02-27 12:51:07.073;2500.0;750.0;1500.0;['Statement', 'S', 'Bastet Noir', 'Mini', 'Dresses', 'Women', 'Green', 'Silk', 'Multi Season'];['Occasion', 'Size', 'Brand', 'Length', 'Category', 'Gender', 'Color', 'Material', 'Seasons']

outfi

In [6]:
outfits = pd.read_csv(rf"{ROOT_PRESS}\outfits_10k.csv", sep=';', on_bad_lines='skip')


In [7]:
import ast

# Chuyển đổi chuỗi thành danh sách
outfits['outfit_tags'] = outfits['outfit_tags'].apply(ast.literal_eval)
outfits['tag_categories'] = outfits['tag_categories'].apply(ast.literal_eval)


In [8]:
outfits.info()


<class 'pandas.DataFrame'>
RangeIndex: 2223 entries, 0 to 2222
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              2223 non-null   str    
 1   name            2223 non-null   str    
 2   description     2206 non-null   str    
 3   group           2223 non-null   str    
 4   owner           2223 non-null   str    
 5   timeCreated     2223 non-null   str    
 6   retailPrice     2220 non-null   float64
 7   pricePerWeek    2223 non-null   float64
 8   pricePerMonth   2223 non-null   float64
 9   outfit_tags     2223 non-null   object 
 10  tag_categories  2223 non-null   object 
dtypes: float64(3), object(2), str(6)
memory usage: 191.2+ KB


In [9]:
outfits.head()

,id,name,description,group,owner,timeCreated,retailPrice,pricePerWeek,pricePerMonth,outfit_tags,tag_categories
0,outfit.ffd83466cdb84a0dba02339aa0c72f73,Mariposa Earrings AB-Crystal,The Mariposa Earrings are the perfect statemen...,group.4f70fa1707b559c0db341fa53997e52f,o_00037,2019-03-22 11:52:11.000,1600.0,250.0,500.0,"[Jewelry, Cecilie Melli, Statement, Metallic, ...","[Category, Brand, Occasion, Details, Occasion]"
1,outfit.ff08dd87bf144defaa332eac0bf5bd4e,The Gisele Dress,This mini satin dress is made in a beautiful o...,group.a65e95a2f1823633a8d0b07b7fff2e7b,o_00740,2020-02-27 12:51:07.073,2500.0,750.0,1500.0,"[Statement, S, Bastet Noir, Mini, Dresses, Wom...","[Occasion, Size, Brand, Length, Category, Gend..."
2,outfit.fec66293d37940b79dde2c1acd761fe3,Evie Deauville Mauve Long Sleeve Dress,The Evie Long Sleeve Dress is made in a flowy ...,group.fcc139e28abeaf451c19d46bf311fcb0,o_00530,2019-09-23 09:47:20.000,2200.0,660.0,1320.0,"[Ruffles, Women, Midi, Purple, Multi Season, S...","[Details, Gender, Length, Color, Seasons, Mate..."
3,outfit.feaa16af0f6b4a96b65f8a29ace77979,Tahiti Earrings Green/Gold,The Tahiti Earrings are the perfect statement ...,group.d940849fd6a77efb5fd69121e43bb3ad,o_00037,2019-03-22 11:34:50.000,1600.0,590.0,1180.0,"[Multi Season, Jewelry, Formal, Green, Metalli...","[Seasons, Category, Occasion, Color, Details, ..."
4,outfit.fea3bb7d8ff54872ad84977465e29da4,Asbjorg Aqua Haze Leopard Skirt,Own this look for a vintage and everyday look....,group.fc687f43497199532a11b0f991ebee04,o_00530,2019-05-21 14:33:03.000,1700.0,590.0,1180.0,"[Everyday, Midi, Ruffles, Wrap, Skirts, Women,...","[Occasion, Length, Details, Fit, Category, Gen..."


In [10]:
outfits['outfit_tags'][0][0]

'Jewelry'

In [11]:
outfits['group'][0]

'group.4f70fa1707b559c0db341fa53997e52f'

In [12]:
outfits.isnull().sum()

id                 0
name               0
description       17
group              0
owner              0
timeCreated        0
retailPrice        3
pricePerWeek       0
pricePerMonth      0
outfit_tags        0
tag_categories     0
dtype: int64

In [13]:
print(outfits[['name', 'description']].isnull().sum())


name            0
description    17
dtype: int64


In [14]:
# Lấy các dòng có cột 'name' là null
outfits_null_name = outfits[outfits['name'].isnull()]

# Hiển thị kết quả
outfits_null_name


,id,name,description,group,owner,timeCreated,retailPrice,pricePerWeek,pricePerMonth,outfit_tags,tag_categories


In [15]:
null_outfit_ids = outfits[outfits['name'].isnull()]['id']
null_outfit_ids

Series([], Name: id, dtype: str)

với cột name loại bỏ missing

In [16]:
outfits = outfits.dropna(subset=['name'])


vơí cột des giữ nguyên thay missing bằng "No description available" nghĩa là không mô tả khả dụng, đảm bảo đầu vào cho mô hình ( tùy trường hợp )

In [17]:
outfits['description'] = outfits['description'].fillna('No description available')


In [18]:
outfits.reset_index(drop=True, inplace=True)  # Đặt lại chỉ mục

In [19]:
outfits.isnull().sum()

id                0
name              0
description       0
group             0
owner             0
timeCreated       0
retailPrice       3
pricePerWeek      0
pricePerMonth     0
outfit_tags       0
tag_categories    0
dtype: int64

In [20]:
outfits.info()


<class 'pandas.DataFrame'>
RangeIndex: 2223 entries, 0 to 2222
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              2223 non-null   str    
 1   name            2223 non-null   str    
 2   description     2223 non-null   str    
 3   group           2223 non-null   str    
 4   owner           2223 non-null   str    
 5   timeCreated     2223 non-null   str    
 6   retailPrice     2220 non-null   float64
 7   pricePerWeek    2223 non-null   float64
 8   pricePerMonth   2223 non-null   float64
 9   outfit_tags     2223 non-null   object 
 10  tag_categories  2223 non-null   object 
dtypes: float64(3), object(2), str(6)
memory usage: 191.2+ KB


In [21]:
outfits.describe()

,retailPrice,pricePerWeek,pricePerMonth
count,2220.000000,2223.000000,2223.000000
mean,3401.374775,709.781826,1396.914530
std,4231.879141,197.717988,430.434391
min,266.000000,150.000000,200.000000
25%,1600.000000,590.000000,1180.000000
50%,2300.000000,630.000000,1260.000000
75%,3500.000000,900.000000,1800.000000
max,80000.000000,1300.000000,2600.000000


In [22]:
outfits.head()

,id,name,description,group,owner,timeCreated,retailPrice,pricePerWeek,pricePerMonth,outfit_tags,tag_categories
0,outfit.ffd83466cdb84a0dba02339aa0c72f73,Mariposa Earrings AB-Crystal,The Mariposa Earrings are the perfect statemen...,group.4f70fa1707b559c0db341fa53997e52f,o_00037,2019-03-22 11:52:11.000,1600.0,250.0,500.0,"[Jewelry, Cecilie Melli, Statement, Metallic, ...","[Category, Brand, Occasion, Details, Occasion]"
1,outfit.ff08dd87bf144defaa332eac0bf5bd4e,The Gisele Dress,This mini satin dress is made in a beautiful o...,group.a65e95a2f1823633a8d0b07b7fff2e7b,o_00740,2020-02-27 12:51:07.073,2500.0,750.0,1500.0,"[Statement, S, Bastet Noir, Mini, Dresses, Wom...","[Occasion, Size, Brand, Length, Category, Gend..."
2,outfit.fec66293d37940b79dde2c1acd761fe3,Evie Deauville Mauve Long Sleeve Dress,The Evie Long Sleeve Dress is made in a flowy ...,group.fcc139e28abeaf451c19d46bf311fcb0,o_00530,2019-09-23 09:47:20.000,2200.0,660.0,1320.0,"[Ruffles, Women, Midi, Purple, Multi Season, S...","[Details, Gender, Length, Color, Seasons, Mate..."
3,outfit.feaa16af0f6b4a96b65f8a29ace77979,Tahiti Earrings Green/Gold,The Tahiti Earrings are the perfect statement ...,group.d940849fd6a77efb5fd69121e43bb3ad,o_00037,2019-03-22 11:34:50.000,1600.0,590.0,1180.0,"[Multi Season, Jewelry, Formal, Green, Metalli...","[Seasons, Category, Occasion, Color, Details, ..."
4,outfit.fea3bb7d8ff54872ad84977465e29da4,Asbjorg Aqua Haze Leopard Skirt,Own this look for a vintage and everyday look....,group.fc687f43497199532a11b0f991ebee04,o_00530,2019-05-21 14:33:03.000,1700.0,590.0,1180.0,"[Everyday, Midi, Ruffles, Wrap, Skirts, Women,...","[Occasion, Length, Details, Fit, Category, Gen..."


# Xử lý picture_triplets

In [24]:
with open(rf"{ROOT_PRESS}\picture_triplets_10k.csv", 'r') as f:
    lines = f.readlines()

# In các dòng đầu tiên để kiểm tra định dạng
for i in range(5):
    print(lines[i])

picture.id;outfit.id;displayOrder;file_name

picture.0000cdba64314d84a49ed1c266589cc0;outfit.794483397da8425a813301eecf9828c6;0;0000cdba64314d84a49ed1c266589cc0.jpg

picture.0010c2e161154d6893734981d5455e76;outfit.9387d05b47f906c5;0;0010c2e161154d6893734981d5455e76.jpg

picture.0013342f663843d4b7397b4b88b98e6d;outfit.b24fc57f02ac4e4c9061761ca18d9d3c;0;0013342f663843d4b7397b4b88b98e6d.jpg

picture.001a58ea68da426384567b8cccc0c8a6;outfit.e989b8cb4a814d97b642e1cb326f47e6;0;001a58ea68da426384567b8cccc0c8a6.jpg



In [25]:

pictures= pd.read_csv(rf"{ROOT_PRESS}\picture_triplets_10k.csv", sep=';', on_bad_lines='skip')



In [26]:
pictures.info()

<class 'pandas.DataFrame'>
RangeIndex: 2223 entries, 0 to 2222
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   picture.id    2223 non-null   str  
 1   outfit.id     2223 non-null   str  
 2   displayOrder  2223 non-null   int64
 3   file_name     2223 non-null   str  
dtypes: int64(1), str(3)
memory usage: 69.6 KB


In [27]:
pictures.head()

,picture.id,outfit.id,displayOrder,file_name
0,picture.0000cdba64314d84a49ed1c266589cc0,outfit.794483397da8425a813301eecf9828c6,0,0000cdba64314d84a49ed1c266589cc0.jpg
1,picture.0010c2e161154d6893734981d5455e76,outfit.9387d05b47f906c5,0,0010c2e161154d6893734981d5455e76.jpg
2,picture.0013342f663843d4b7397b4b88b98e6d,outfit.b24fc57f02ac4e4c9061761ca18d9d3c,0,0013342f663843d4b7397b4b88b98e6d.jpg
3,picture.001a58ea68da426384567b8cccc0c8a6,outfit.e989b8cb4a814d97b642e1cb326f47e6,0,001a58ea68da426384567b8cccc0c8a6.jpg
4,picture.0021a36dd3b04f7ebff9430b7801d44d,outfit.070bc7bd72154b5a805f52f086fb72ed,0,0021a36dd3b04f7ebff9430b7801d44d.jpg


In [28]:
null_outfit_ids

Series([], Name: id, dtype: str)

In [29]:
# Bước 2: Loại bỏ các giao dịch liên quan đến các outfit.id đó
pictures_filter = pictures[pictures['outfit.id'].isin(null_outfit_ids)]
pictures_filter

,picture.id,outfit.id,displayOrder,file_name


In [30]:
# Bước 2: Loại bỏ các giao dịch liên quan đến các outfit.id đó
pictures = pictures[~pictures['outfit.id'].isin(null_outfit_ids)]

In [31]:
pictures.info()

<class 'pandas.DataFrame'>
RangeIndex: 2223 entries, 0 to 2222
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   picture.id    2223 non-null   str  
 1   outfit.id     2223 non-null   str  
 2   displayOrder  2223 non-null   int64
 3   file_name     2223 non-null   str  
dtypes: int64(1), str(3)
memory usage: 69.6 KB


# Xử lý user_activity_triplets

In [33]:
with open(rf"{ROOT_PRESS}\\user_activity_triplets_10k.csv", 'r') as f:
    lines = f.readlines()

# In các dòng đầu tiên để kiểm tra định dạng
for i in range(5):
    print(lines[i])

customer.id;outfit.id;rentalPeriod.start;rentalPeriod.end

1128;outfit.03bdf90c39e1431685d49ce8309dd242;2021-10-11;2021-11-11

1128;outfit.c46affd5f8ee4749b1f8e7d4ff47f3ff;2021-10-11;2021-11-11

1128;outfit.900dd323358401b1;2022-02-22;2022-03-11

1128;outfit.0630ef7d996d4c489b4e54e85bd09e5a;2021-11-11;2021-12-15



In [34]:

transactions = pd.read_csv(rf"{ROOT_PRESS}\\user_activity_triplets_10k.csv", sep=';', on_bad_lines='skip')



In [35]:
transactions['rentalPeriod.start'] = pd.to_datetime(transactions['rentalPeriod.start'])
transactions['rentalPeriod.end'] = pd.to_datetime(transactions['rentalPeriod.end'])


In [36]:
transactions.info()


<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   customer.id         10000 non-null  int64         
 1   outfit.id           10000 non-null  str           
 2   rentalPeriod.start  10000 non-null  datetime64[us]
 3   rentalPeriod.end    10000 non-null  datetime64[us]
dtypes: datetime64[us](2), int64(1), str(1)
memory usage: 312.6 KB


In [37]:
transactions.head()

,customer.id,outfit.id,rentalPeriod.start,rentalPeriod.end
0,1128,outfit.03bdf90c39e1431685d49ce8309dd242,2021-10-11,2021-11-11
1,1128,outfit.c46affd5f8ee4749b1f8e7d4ff47f3ff,2021-10-11,2021-11-11
2,1128,outfit.900dd323358401b1,2022-02-22,2022-03-11
3,1128,outfit.0630ef7d996d4c489b4e54e85bd09e5a,2021-11-11,2021-12-15
4,6310,outfit.bd37bc0ed1857fb5,2022-07-07,2022-08-06


In [38]:
null_outfit_ids

Series([], Name: id, dtype: str)

In [39]:
# Bước 2: Loại bỏ các giao dịch liên quan đến các outfit.id đó
transactions_filter = transactions[transactions['outfit.id'].isin(null_outfit_ids)]
transactions_filter

,customer.id,outfit.id,rentalPeriod.start,rentalPeriod.end


In [40]:

# Bước 2: Loại bỏ các giao dịch liên quan đến các outfit.id đó
transactions_cleaned = transactions[~transactions['outfit.id'].isin(null_outfit_ids)]

In [41]:
transactions_cleaned.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   customer.id         10000 non-null  int64         
 1   outfit.id           10000 non-null  str           
 2   rentalPeriod.start  10000 non-null  datetime64[us]
 3   rentalPeriod.end    10000 non-null  datetime64[us]
dtypes: datetime64[us](2), int64(1), str(1)
memory usage: 312.6 KB


# Preprocessing 

In [42]:
outfits.info()

<class 'pandas.DataFrame'>
RangeIndex: 2223 entries, 0 to 2222
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              2223 non-null   str    
 1   name            2223 non-null   str    
 2   description     2223 non-null   str    
 3   group           2223 non-null   str    
 4   owner           2223 non-null   str    
 5   timeCreated     2223 non-null   str    
 6   retailPrice     2220 non-null   float64
 7   pricePerWeek    2223 non-null   float64
 8   pricePerMonth   2223 non-null   float64
 9   outfit_tags     2223 non-null   object 
 10  tag_categories  2223 non-null   object 
dtypes: float64(3), object(2), str(6)
memory usage: 191.2+ KB


In [43]:
# Số phần tử không trùng lặp trong cột 'column_name'
unique_count = outfits['id'].nunique()
print(f"Số phần tử không trùng lặpe: {unique_count}")


Số phần tử không trùng lặpe: 2223


In [44]:
transactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   customer.id         10000 non-null  int64         
 1   outfit.id           10000 non-null  str           
 2   rentalPeriod.start  10000 non-null  datetime64[us]
 3   rentalPeriod.end    10000 non-null  datetime64[us]
dtypes: datetime64[us](2), int64(1), str(1)
memory usage: 312.6 KB


In [45]:
# Số phần tử không trùng lặp trong cột 'column_name'
unique_count = transactions['outfit.id'].nunique()
print(f"Số phần tử không trùng lặp: {unique_count}")


Số phần tử không trùng lặp: 2223


In [46]:
pictures.info()

<class 'pandas.DataFrame'>
RangeIndex: 2223 entries, 0 to 2222
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   picture.id    2223 non-null   str  
 1   outfit.id     2223 non-null   str  
 2   displayOrder  2223 non-null   int64
 3   file_name     2223 non-null   str  
dtypes: int64(1), str(3)
memory usage: 69.6 KB


In [47]:
# Số phần tử không trùng lặp trong cột 'column_name'
unique_count = pictures['outfit.id'].nunique()
print(f"Số phần tử không trùng lặp: {unique_count}")


Số phần tử không trùng lặp: 2223


In [48]:
# Lấy danh sách các outfit.id từ outfits và transactions
outfit_ids = set(outfits['id'])  # Cột 'id' từ outfits
transaction_outfit_ids = set(transactions['outfit.id'])  # Cột 'outfit.id' từ transactions

# Tìm các id có trong transactions nhưng không có trong outfits
foreign_ids = transaction_outfit_ids - outfit_ids

# Kết quả
if foreign_ids:
    print("Các outfit.id ngoại lai (có trong transactions nhưng không có trong outfits):")
    print(foreign_ids)
else:
    print("Không có outfit.id ngoại lai.")


Không có outfit.id ngoại lai.


In [49]:
# Lấy danh sách các outfit.id từ outfits và transactions
pictures_ids = set(pictures['outfit.id'])  # Cột 'id' từ outfits
transaction_outfit_ids = set(transactions['outfit.id'])  # Cột 'outfit.id' từ transactions

# Tìm các id có trong transactions nhưng không có trong outfits
foreign_ids = transaction_outfit_ids - pictures_ids

# Kết quả
if foreign_ids:
    print("Các outfit.id (trong pictures ) ngoại lai (có trong transactions nhưng không có trong outfits):")
    print(foreign_ids)
else:
    print("Không có outfit.id (trong pictures ) ngoại lai.")


Không có outfit.id (trong pictures ) ngoại lai.


In [249]:
transactions = transactions[transactions['outfit.id'].isin(pictures['outfit.id'])]

In [50]:
# Kiểm tra các id có trong transactions nhưng không có trong outfits
foreign_ids_df = transactions[~transactions['outfit.id'].isin(outfits['id'])]

# Kết quả
if not foreign_ids_df.empty:
    print("Các giao dịch có outfit.id không tồn tại trong outfits:")
    print(foreign_ids_df)
else:
    print("Tất cả outfit.id trong transactions đều có trong outfits.")


Tất cả outfit.id trong transactions đều có trong outfits.


In [51]:
# Kiểm tra các id có trong transactions nhưng không có trong pictures
foreign_ids_df = transactions[~transactions['outfit.id'].isin(pictures['outfit.id'])]

# Kết quả
if not foreign_ids_df.empty:
    print("Các giao dịch có outfit.id không tồn tại trong pictures:")
    print(foreign_ids_df)
else:
    print("Tất cả outfit.id trong transactions đều có trong pictures.")

Tất cả outfit.id trong transactions đều có trong pictures.


In [63]:
import time

# Đổi tên cột
data_collect = transactions.rename(columns={
    "outfit.id": "item_id_original",
    "customer.id": "user_id_original",
    "rentalPeriod.start": "time"
})

# Chuyển cột "time" thành UNIX Timestamp (Epoch Time)
data_collect["time"] = pd.to_datetime(data_collect["time"])
data_collect["time"] = data_collect["time"].apply(lambda x: int(time.mktime(x.timetuple())))


data_collect = data_collect[['user_id_original', 'item_id_original' , 'time']]

# Lưu DataFrame đã xử lý thành file CSV mới
output_path = rf"{ROOT}\dataset_VCR.csv"
data_collect.to_csv(output_path, index=False)
    
# Hiển thị dữ liệu đã xử lý
data_collect.head()


,user_id_original,item_id_original,time
0,1128,outfit.03bdf90c39e1431685d49ce8309dd242,1633885200
1,1128,outfit.c46affd5f8ee4749b1f8e7d4ff47f3ff,1633885200
2,1128,outfit.900dd323358401b1,1645462800
3,1128,outfit.0630ef7d996d4c489b4e54e85bd09e5a,1636563600
4,6310,outfit.bd37bc0ed1857fb5,1657126800


In [64]:
data_collect.shape

(10000, 3)

In [65]:
data_collect["user_id_original"].nunique()

711

In [66]:
data_collect["item_id_original"].nunique()

2223

# Xử lý default df

In [67]:
# Đọc file CSV
df = pd.read_csv(rf"{ROOT}\dataset_VCR.csv")

# Lấy 3 cột chính
df

,user_id_original,item_id_original,time
0,1128,outfit.03bdf90c39e1431685d49ce8309dd242,1633885200
1,1128,outfit.c46affd5f8ee4749b1f8e7d4ff47f3ff,1633885200
2,1128,outfit.900dd323358401b1,1645462800
3,1128,outfit.0630ef7d996d4c489b4e54e85bd09e5a,1636563600
4,6310,outfit.bd37bc0ed1857fb5,1657126800
...,...,...,...
9995,3189,outfit.706635d443e040d2a4ae9e6862a95961,1663952400
9996,3320,outfit.e0f170b8b51f4ae4a7e00047b9ea8d79,1665248400
9997,7376,outfit.9bd9226c247f4ee8a64be90ee310fd8b,1626627600
9998,5837,outfit.bb35c791b2bfa191,1619974800


In [68]:
df.shape

(10000, 3)

In [69]:
df['user_id_original'].value_counts()

user_id_original
3267    126
6907    124
3094    116
675     103
7411     93
       ... 
2788      3
1211      3
1680      3
747       3
2219      3
Name: count, Length: 711, dtype: int64

In [70]:
df['item_id_original'].value_counts()

item_id_original
outfit.b4e582b38a10f6bf                    21
outfit.aa76777d3b9d4073a342e2093e4ea1e2    20
outfit.2ea36f84cb12427686eed685f97f2a8f    19
outfit.b45f71cef3bf96b3                    19
outfit.fd816aa2311d45c78fd0ab2d663a177a    18
                                           ..
outfit.e5573a0b879d4a31b02ff8dfe70ea20f     1
outfit.0ac1ac7d05ed43eeb9cb12c66cae3d18     1
outfit.092ee39646f34dafac13607ee3c17215     1
outfit.296da213b6db4d99817dea16407654f0     1
outfit.776c13028a5e48b7899f4a9057ee8e18     1
Name: count, Length: 2223, dtype: int64

In [77]:


def process_data(df, random_percent=0.1, n_core=10, random_state=42):
    """
    Xử lý dataframe theo quy trình
    1. Lấy random k% số phần tử và lưu lại 
       - random_percent: phần trăm dữ liệu muốn lấy (0-1)
       - random_state: tái tạo kết quả
    2. Áp dụng n-core
       - n_core: số lượng tương tác tối thiểu
    3. Mapping ID cho User và Item
    4. Chia thành train/val/test
    
    """
    BASE = r"E:\DoCode\CD2\source\Source\get_hrs_rs\rs\get10k_data\output_10k_sample"
    # 1. Lấy random k% số phần tử
    np.random.seed(random_state)
    n_samples = int(len(df) * random_percent)
    sampled_df = df.sample(n=n_samples, random_state=random_state)
    
    # Lưu file csv với k% phần tử
    # sampled_df.to_csv(f'dataset_VCR_{random_percent}_{random_state}.csv', index=False)
    
    # 2. Áp dụng n-core
    # Đếm số lượng tương tác của mỗi user và item
    user_counts = sampled_df['user_id_original'].value_counts()
    # item_counts = sampled_df['item_id_original'].value_counts()
    
    # Lọc các user và item có ít nhất n tương tác
    valid_users = user_counts[user_counts >= n_core].index
    # valid_items = item_counts[item_counts >= n_core].index
    
    # filtered_df = sampled_df[
    #     sampled_df['user_id_original'].isin(valid_users) & 
    #     sampled_df['item_id_original'].isin(valid_items)
    # ]
    
    # Giữ lại dữ liệu của các user hợp lệ
    filtered_df = sampled_df[sampled_df['user_id_original'].isin(valid_users)]
    
    # 3. Mapping ID
    # Tạo mapping cho user
    unique_users = filtered_df['user_id_original'].unique()
    user_id_map = {old_id: new_id for new_id, old_id in enumerate(unique_users , start=0)}
    
    # Tạo mapping cho item
    unique_items = filtered_df['item_id_original'].unique()
    item_id_map = {old_id: new_id for new_id, old_id in enumerate(unique_items , start=0)}
    
    # Áp dụng mapping
    mapped_df = filtered_df.copy()
    mapped_df['user_id'] = mapped_df['user_id_original'].map(user_id_map)
    mapped_df['item_id'] = mapped_df['item_id_original'].map(item_id_map)

    print(mapped_df.head())
    # Lưu file csv sau mapping
    out_path = rf"{BASE}\dataset_VCR_{random_percent}_{random_state}_{n_core}.csv"
    mapped_df.to_csv(out_path, index=False)
    print(f"✅ Đã lưu: {out_path}")
    print(f"   Users: {mapped_df['user_id'].nunique()}")
    print(f"   Items: {mapped_df['item_id'].nunique()}")
    print(f"   Records: {len(mapped_df)}")
    
    # # 4. Chia thành train/val/test
    # # Sắp xếp theo thời gian
    # mapped_df = mapped_df.sort_values('time')
    
    # # Chia theo tỷ lệ 70/15/15
    # train_df, temp_df = train_test_split(mapped_df, test_size=0.3, shuffle=False)
    # val_df, test_df = train_test_split(temp_df, test_size=0.5, shuffle=False)
    
    # Lưu các file
    # train_df.to_csv(f'dataset_VCR_{random_percent}_{random_state}_{n_core}_train.csv', index=False)
    # val_df.to_csv(f'dataset_VCR_{random_percent}_{random_state}_{n_core}_val.csv', index=False)
    # test_df.to_csv(f'dataset_VCR_{random_percent}_{random_state}_{n_core}_test.csv', index=False)
    
    # return train_df, val_df, test_df, user_id_map, item_id_map

# Sử dụng hàm
process_data(
    df, 
    random_percent=1,  # Lấy 20% dữ liệu
    n_core=5,           # Loại bỏ user/item có ít hơn 5 tương tác
    random_state=42     # Seed để tái tạo kết quả
)

# In thông tin về kết quả
# print(f"Số lượng users sau khi mapping: {len(user_mapping)}")
# print(f"Số lượng items sau khi mapping: {len(item_mapping)}")
# print("\nKích thước các tập dữ liệu:")
# print(f"Train: {len(train)} rows")
# print(f"Validation: {len(val)} rows")
# print(f"Test: {len(test)} rows")

      user_id_original                         item_id_original        time  \
6252               446  outfit.7cc6bd8e645a41d2837976a0d09e3ca9  1570467600   
4684              7367  outfit.7ee7542297f944ecb49f4fe5b44728f7  1680454800   
1731              6696  outfit.e8e4d130fa3e4f108127dc74fd854a75  1621270800   
4742              1878  outfit.7395b78d10084aebb906f8f83bfd89cc  1658854800   
4521              4906  outfit.c929e1bd6aa940f7a34c16471389c460  1596042000   

      user_id  item_id  
6252        0        0  
4684        1        1  
1731        2        2  
4742        3        3  
4521        4        4  
✅ Đã lưu: E:\DoCode\CD2\source\Source\get_hrs_rs\rs\get10k_data\output_10k_sample\dataset_VCR_1_42_5.csv
   Users: 553
   Items: 2194
   Records: 9455


# Xử lý sample df

In [78]:
# Đọc file CSV
df = pd.read_csv(rf"{ROOT}\dataset_VCR_1_42_5.csv")
# "E:\DoCode\CD2\source\Source\get_hrs_rs\rs\get10k_data\preprocess\dataset_VCR_0.5_42_10.csv"
# df = pd.read_csv(r'/kaggle/working/dataset_VCR_full_10.csv')

In [79]:
df.head(10)

,user_id_original,item_id_original,time,user_id,item_id
0,446,outfit.7cc6bd8e645a41d2837976a0d09e3ca9,1570467600,0,0
1,7367,outfit.7ee7542297f944ecb49f4fe5b44728f7,1680454800,1,1
2,6696,outfit.e8e4d130fa3e4f108127dc74fd854a75,1621270800,2,2
3,1878,outfit.7395b78d10084aebb906f8f83bfd89cc,1658854800,3,3
4,4906,outfit.c929e1bd6aa940f7a34c16471389c460,1596042000,4,4
5,7411,outfit.6a8a9e4b00374d0e953550820b496573,1603990800,5,5
6,1374,outfit.1e4a3ce9976a4d9a9061c167255f3875,1601830800,6,6
7,6118,outfit.5862a9386cbc4ff29e9892f49274c243,1570467600,7,7
8,7101,outfit.f6e0ac328eac45939027a5dbf9d8a086,1618938000,8,8
9,5929,outfit.69510e71049843628dabe5ef896f1b04,1636909200,9,9


In [264]:
# df = df[['user_id' , 'item_id' , 'time']]

In [80]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9455 entries, 0 to 9454
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   user_id_original  9455 non-null   int64
 1   item_id_original  9455 non-null   str  
 2   time              9455 non-null   int64
 3   user_id           9455 non-null   int64
 4   item_id           9455 non-null   int64
dtypes: int64(4), str(1)
memory usage: 369.5 KB


In [81]:
# Sắp xếp dữ liệu theo `user_id` và `time`
df = df.sort_values(by=["user_id", "time"]).reset_index(drop=True)


In [82]:
df.head(10)

,user_id_original,item_id_original,time,user_id,item_id
0,446,outfit.9edf55f5369fd779,1570381200,0,1906
1,446,outfit.7cc6bd8e645a41d2837976a0d09e3ca9,1570467600,0,0
2,446,outfit.5b219e65b35544769ae32989b15ff364,1570467600,0,1127
3,446,outfit.989e3eb7e1be092f,1570467600,0,1695
4,446,outfit.84f134b06fbed5a0,1570467600,0,960
5,446,outfit.9aadfc9a1d195c33,1570467600,0,1013
6,446,outfit.af4c1536e8d62d77,1571936400,0,1483
7,446,outfit.dd6c85fa43ab4ef7989ebb6a72302ca0,1571936400,0,1930
8,446,outfit.d6578bc3024d48369705dc09bd28f070,1571936400,0,2049
9,446,outfit.925c69db26f6bbae,1571936400,0,367


In [83]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9455 entries, 0 to 9454
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   user_id_original  9455 non-null   int64
 1   item_id_original  9455 non-null   str  
 2   time              9455 non-null   int64
 3   user_id           9455 non-null   int64
 4   item_id           9455 non-null   int64
dtypes: int64(4), str(1)
memory usage: 369.5 KB


# Tạo user list

In [84]:
user_df = df[['user_id_original', 'user_id']]

In [85]:
user_df = user_df.drop_duplicates()

In [95]:

# Sắp xếp theo giá trị user_id tăng dần
user_list_df = user_df.sort_values(by='user_id')

# Ghi vào file user_list.txt, không bao gồm dòng tên cột
user_list_df.to_csv(rf"{ROOT}\\user_list.txt", sep=' ', index=False, header=False)

print("File user_list.txt đã được tạo.")


File user_list.txt đã được tạo.


In [96]:
user_list_df.info()

<class 'pandas.DataFrame'>
Index: 553 entries, 0 to 9450
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   user_id_original  553 non-null    int64
 1   user_id           553 non-null    int64
dtypes: int64(2)
memory usage: 13.0 KB


# Tạo item list

In [97]:
item_df = df[['item_id_original', 'item_id']]

In [98]:
item_df = item_df.drop_duplicates()

In [99]:
# Sắp xếp theo giá trị user_id tăng dần
item_list_df = item_df.sort_values(by='item_id')

# Ghi vào file user_list.txt, không bao gồm dòng tên cột
item_list_df.to_csv(rf"{ROOT}\\item_list.txt", sep=' ', index=False, header=False)

print("File item_list.txt đã được tạo.")

File item_list.txt đã được tạo.


In [100]:
item_list_df.info()

<class 'pandas.DataFrame'>
Index: 2194 entries, 1 to 1453
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   item_id_original  2194 non-null   str  
 1   item_id           2194 non-null   int64
dtypes: int64(1), str(1)
memory usage: 51.4 KB


In [101]:
item_list_df.head()

,item_id_original,item_id
1,outfit.7cc6bd8e645a41d2837976a0d09e3ca9,0
51,outfit.7ee7542297f944ecb49f4fe5b44728f7,1
56,outfit.e8e4d130fa3e4f108127dc74fd854a75,2
83,outfit.7395b78d10084aebb906f8f83bfd89cc,3
91,outfit.c929e1bd6aa940f7a34c16471389c460,4


# Tạo interactions matrix

In [102]:
# Tạo dictionary lưu quan hệ user và danh sách item
user_item_interactions = df.groupby('user_id')['item_id'].apply(lambda x: sorted(x)).to_dict()

# Ghi file intersection_user.txt
with open(rf"{ROOT}\\intersection_user.txt", 'w') as f:
    for user_id in sorted(user_item_interactions.keys()):  # Sắp xếp user_id theo thứ tự tăng dần
        # Lấy danh sách item_id đã được sắp xếp và chuyển thành chuỗi cách nhau bởi khoảng trắng
        item_list_str = ' '.join(map(str, user_item_interactions[user_id]))
        # Ghi user_id và danh sách item vào file
        f.write(f"{user_id} {item_list_str}\n")

print("File intersection_user.txt đã được tạo.")


File intersection_user.txt đã được tạo.


## **=> Kiểm tra lại dữ liệu**

In [103]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9455 entries, 0 to 9454
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   user_id_original  9455 non-null   int64
 1   item_id_original  9455 non-null   str  
 2   time              9455 non-null   int64
 3   user_id           9455 non-null   int64
 4   item_id           9455 non-null   int64
dtypes: int64(4), str(1)
memory usage: 369.5 KB


In [104]:
# Số phần tử không trùng lặp trong cột 'column_name'
unique_count = user_df['user_id_original'].nunique()
print(f"Số phần tử không trùng lặp trong cột user_id_original: {unique_count}")


Số phần tử không trùng lặp trong cột user_id_original: 553


In [105]:
# Số phần tử không trùng lặp trong cột 'column_name'
unique_count = item_df['item_id_original'].nunique()
print(f"Số phần tử không trùng lặp trong cột item_id_original: {unique_count}")


Số phần tử không trùng lặp trong cột item_id_original: 2194


In [106]:
# Số phần tử không trùng lặp trong cột 'column_name'
unique_count = item_df['item_id'].max()
print(f"phần tử lớn nhất trong cột item_id: {unique_count}")

phần tử lớn nhất trong cột item_id: 2193


In [107]:
# Số phần tử không trùng lặp trong cột 'column_name'
unique_count = user_df['user_id'].max()
print(f"phần tử lớn nhất trong cột user_id: {unique_count}")

phần tử lớn nhất trong cột user_id: 552


# Chia train / test 

In [108]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9455 entries, 0 to 9454
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   user_id_original  9455 non-null   int64
 1   item_id_original  9455 non-null   str  
 2   time              9455 non-null   int64
 3   user_id           9455 non-null   int64
 4   item_id           9455 non-null   int64
dtypes: int64(4), str(1)
memory usage: 369.5 KB


In [109]:
df.tail(10)

,user_id_original,item_id_original,time,user_id,item_id
9445,6013,outfit.3ccefaac1825408c82d9980a3ea5964a,1636909200,551,1630
9446,6013,outfit.76443d6db496494dbb26ac275ecbdb3b,1644858000,551,72
9447,6013,outfit.880e71e3568b6812,1649264400,551,1671
9448,6013,outfit.48b43358755242c4bf529574c171b675,1651856400,551,1835
9449,6013,outfit.50a8ff8407894fcb9048249c78732fc4,1654534800,551,1718
9450,1073,outfit.043a430a63b34151a6489d2863ea11aa,1596387600,552,978
9451,1073,outfit.b2f9d48f3d4c4549b8d51fcd04979a79,1596387600,552,1380
9452,1073,outfit.adc57b4886354fdcafea896cdd2609b8,1596387600,552,1970
9453,1073,outfit.88e9a4b11d284db686be6be18502f641,1596387600,552,1493
9454,1073,outfit.6377a711bdf04683819c6fc08921cfaf,1596387600,552,2004


In [110]:

# Thêm cột `rank` cho từng `user_id`
df["rank"] = df.groupby("user_id").cumcount() + 1


In [111]:
# Tính toán số lượng tương tác và ngưỡng phân chia
user_counts = df.groupby("user_id")["rank"].max().reset_index()
user_counts.rename(columns={"rank": "total_interactions"}, inplace=True)
user_counts["train_threshold"] = (user_counts["total_interactions"] * 0.8).astype(int)


In [112]:
# Gộp ngưỡng với DataFrame ban đầu
df = df.merge(user_counts, on="user_id", how="inner")


In [113]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9455 entries, 0 to 9454
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   user_id_original    9455 non-null   int64
 1   item_id_original    9455 non-null   str  
 2   time                9455 non-null   int64
 3   user_id             9455 non-null   int64
 4   item_id             9455 non-null   int64
 5   rank                9455 non-null   int64
 6   total_interactions  9455 non-null   int64
 7   train_threshold     9455 non-null   int32
dtypes: int32(1), int64(6), str(1)
memory usage: 554.1 KB


In [114]:
df.head(10)

,user_id_original,item_id_original,time,user_id,item_id,rank,total_interactions,train_threshold
0,446,outfit.9edf55f5369fd779,1570381200,0,1906,1,47,37
1,446,outfit.7cc6bd8e645a41d2837976a0d09e3ca9,1570467600,0,0,2,47,37
2,446,outfit.5b219e65b35544769ae32989b15ff364,1570467600,0,1127,3,47,37
3,446,outfit.989e3eb7e1be092f,1570467600,0,1695,4,47,37
4,446,outfit.84f134b06fbed5a0,1570467600,0,960,5,47,37
5,446,outfit.9aadfc9a1d195c33,1570467600,0,1013,6,47,37
6,446,outfit.af4c1536e8d62d77,1571936400,0,1483,7,47,37
7,446,outfit.dd6c85fa43ab4ef7989ebb6a72302ca0,1571936400,0,1930,8,47,37
8,446,outfit.d6578bc3024d48369705dc09bd28f070,1571936400,0,2049,9,47,37
9,446,outfit.925c69db26f6bbae,1571936400,0,367,10,47,37


In [115]:
df.tail(10)

,user_id_original,item_id_original,time,user_id,item_id,rank,total_interactions,train_threshold
9445,6013,outfit.3ccefaac1825408c82d9980a3ea5964a,1636909200,551,1630,1,5,4
9446,6013,outfit.76443d6db496494dbb26ac275ecbdb3b,1644858000,551,72,2,5,4
9447,6013,outfit.880e71e3568b6812,1649264400,551,1671,3,5,4
9448,6013,outfit.48b43358755242c4bf529574c171b675,1651856400,551,1835,4,5,4
9449,6013,outfit.50a8ff8407894fcb9048249c78732fc4,1654534800,551,1718,5,5,4
9450,1073,outfit.043a430a63b34151a6489d2863ea11aa,1596387600,552,978,1,5,4
9451,1073,outfit.b2f9d48f3d4c4549b8d51fcd04979a79,1596387600,552,1380,2,5,4
9452,1073,outfit.adc57b4886354fdcafea896cdd2609b8,1596387600,552,1970,3,5,4
9453,1073,outfit.88e9a4b11d284db686be6be18502f641,1596387600,552,1493,4,5,4
9454,1073,outfit.6377a711bdf04683819c6fc08921cfaf,1596387600,552,2004,5,5,4


In [116]:
# Tạo ma trận train và test từ DataFrame
train_interactions = {}
test_interactions = {}

for user_id, user_df in df.groupby('user_id'):
    # Lấy ngưỡng train_threshold cho user hiện tại
    train_threshold = user_df['train_threshold'].iloc[0]

    # Phân chia item_id thành train và test
    train_items = user_df[user_df['rank'] <= train_threshold]['item_id'].tolist()
    test_items = user_df[user_df['rank'] > train_threshold]['item_id'].tolist()

    # Lưu vào dictionary
    train_interactions[user_id] = train_items
    test_interactions[user_id] = test_items


In [117]:
len(train_interactions)

553

In [118]:
len(train_interactions[0])

37

In [119]:
len(test_interactions)

553

In [120]:
len(test_interactions[0])

10

In [121]:
# Ghi file train.txt
with open(rf"{ROOT}\\train.txt", 'w') as f:
    for user_id, item_list in train_interactions.items():
        item_list_str = ' '.join(map(str, item_list))  # Chuyển danh sách item_id thành chuỗi
        f.write(f"{user_id} {item_list_str}\n")  # Ghi user_id và item_id cách nhau bởi khoảng trắng

# Ghi file test.txt
with open(rf"{ROOT}\\test.txt", 'w') as f:
    for user_id, item_list in test_interactions.items():
        item_list_str = ' '.join(map(str, item_list))  # Chuyển danh sách item_id thành chuỗi
        f.write(f"{user_id} {item_list_str}\n")  # Ghi user_id và item_id cách nhau bởi khoảng trắng

print("Files train.txt và test.txt đã được tạo.")


Files train.txt và test.txt đã được tạo.


=> đang sai chia lại theo hướng , mỗi người dùng , đều có trong train và test (đã fix)

# Tạo items_features

## Xử lý feature1 và 2

In [122]:
item_df.info()

<class 'pandas.DataFrame'>
Index: 2194 entries, 0 to 9438
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   item_id_original  2194 non-null   str  
 1   item_id           2194 non-null   int64
dtypes: int64(1), str(1)
memory usage: 51.4 KB


In [123]:
item_df.reset_index(drop=True, inplace=True)  # Đặt lại chỉ mục

In [124]:
item_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2194 entries, 0 to 2193
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   item_id_original  2194 non-null   str  
 1   item_id           2194 non-null   int64
dtypes: int64(1), str(1)
memory usage: 34.4 KB


In [125]:
item_df.head()

,item_id_original,item_id
0,outfit.9edf55f5369fd779,1906
1,outfit.7cc6bd8e645a41d2837976a0d09e3ca9,0
2,outfit.5b219e65b35544769ae32989b15ff364,1127
3,outfit.989e3eb7e1be092f,1695
4,outfit.84f134b06fbed5a0,960


In [126]:
outfits.info()

<class 'pandas.DataFrame'>
RangeIndex: 2223 entries, 0 to 2222
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              2223 non-null   str    
 1   name            2223 non-null   str    
 2   description     2223 non-null   str    
 3   group           2223 non-null   str    
 4   owner           2223 non-null   str    
 5   timeCreated     2223 non-null   str    
 6   retailPrice     2220 non-null   float64
 7   pricePerWeek    2223 non-null   float64
 8   pricePerMonth   2223 non-null   float64
 9   outfit_tags     2223 non-null   object 
 10  tag_categories  2223 non-null   object 
dtypes: float64(3), object(2), str(6)
memory usage: 191.2+ KB


In [127]:
outfits.head()

,id,name,description,group,owner,timeCreated,retailPrice,pricePerWeek,pricePerMonth,outfit_tags,tag_categories
0,outfit.ffd83466cdb84a0dba02339aa0c72f73,Mariposa Earrings AB-Crystal,The Mariposa Earrings are the perfect statemen...,group.4f70fa1707b559c0db341fa53997e52f,o_00037,2019-03-22 11:52:11.000,1600.0,250.0,500.0,"[Jewelry, Cecilie Melli, Statement, Metallic, ...","[Category, Brand, Occasion, Details, Occasion]"
1,outfit.ff08dd87bf144defaa332eac0bf5bd4e,The Gisele Dress,This mini satin dress is made in a beautiful o...,group.a65e95a2f1823633a8d0b07b7fff2e7b,o_00740,2020-02-27 12:51:07.073,2500.0,750.0,1500.0,"[Statement, S, Bastet Noir, Mini, Dresses, Wom...","[Occasion, Size, Brand, Length, Category, Gend..."
2,outfit.fec66293d37940b79dde2c1acd761fe3,Evie Deauville Mauve Long Sleeve Dress,The Evie Long Sleeve Dress is made in a flowy ...,group.fcc139e28abeaf451c19d46bf311fcb0,o_00530,2019-09-23 09:47:20.000,2200.0,660.0,1320.0,"[Ruffles, Women, Midi, Purple, Multi Season, S...","[Details, Gender, Length, Color, Seasons, Mate..."
3,outfit.feaa16af0f6b4a96b65f8a29ace77979,Tahiti Earrings Green/Gold,The Tahiti Earrings are the perfect statement ...,group.d940849fd6a77efb5fd69121e43bb3ad,o_00037,2019-03-22 11:34:50.000,1600.0,590.0,1180.0,"[Multi Season, Jewelry, Formal, Green, Metalli...","[Seasons, Category, Occasion, Color, Details, ..."
4,outfit.fea3bb7d8ff54872ad84977465e29da4,Asbjorg Aqua Haze Leopard Skirt,Own this look for a vintage and everyday look....,group.fc687f43497199532a11b0f991ebee04,o_00530,2019-05-21 14:33:03.000,1700.0,590.0,1180.0,"[Everyday, Midi, Ruffles, Wrap, Skirts, Women,...","[Occasion, Length, Details, Fit, Category, Gen..."


In [128]:
item_df['item_id_original'].nunique()

2194

In [129]:
outfits['id'].nunique()

2223

In [130]:
merged_df = pd.merge(outfits, item_df, left_on='id', right_on='item_id_original', how='inner')

In [131]:
merged_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2194 entries, 0 to 2193
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                2194 non-null   str    
 1   name              2194 non-null   str    
 2   description       2194 non-null   str    
 3   group             2194 non-null   str    
 4   owner             2194 non-null   str    
 5   timeCreated       2194 non-null   str    
 6   retailPrice       2191 non-null   float64
 7   pricePerWeek      2194 non-null   float64
 8   pricePerMonth     2194 non-null   float64
 9   outfit_tags       2194 non-null   object 
 10  tag_categories    2194 non-null   object 
 11  item_id_original  2194 non-null   str    
 12  item_id           2194 non-null   int64  
dtypes: float64(3), int64(1), object(2), str(7)
memory usage: 223.0+ KB


In [132]:
merged_df['id'].nunique()

2194

In [133]:
merged_df.head()

,id,name,description,group,owner,timeCreated,retailPrice,pricePerWeek,pricePerMonth,outfit_tags,tag_categories,item_id_original,item_id
0,outfit.ffd83466cdb84a0dba02339aa0c72f73,Mariposa Earrings AB-Crystal,The Mariposa Earrings are the perfect statemen...,group.4f70fa1707b559c0db341fa53997e52f,o_00037,2019-03-22 11:52:11.000,1600.0,250.0,500.0,"[Jewelry, Cecilie Melli, Statement, Metallic, ...","[Category, Brand, Occasion, Details, Occasion]",outfit.ffd83466cdb84a0dba02339aa0c72f73,482
1,outfit.ff08dd87bf144defaa332eac0bf5bd4e,The Gisele Dress,This mini satin dress is made in a beautiful o...,group.a65e95a2f1823633a8d0b07b7fff2e7b,o_00740,2020-02-27 12:51:07.073,2500.0,750.0,1500.0,"[Statement, S, Bastet Noir, Mini, Dresses, Wom...","[Occasion, Size, Brand, Length, Category, Gend...",outfit.ff08dd87bf144defaa332eac0bf5bd4e,580
2,outfit.fec66293d37940b79dde2c1acd761fe3,Evie Deauville Mauve Long Sleeve Dress,The Evie Long Sleeve Dress is made in a flowy ...,group.fcc139e28abeaf451c19d46bf311fcb0,o_00530,2019-09-23 09:47:20.000,2200.0,660.0,1320.0,"[Ruffles, Women, Midi, Purple, Multi Season, S...","[Details, Gender, Length, Color, Seasons, Mate...",outfit.fec66293d37940b79dde2c1acd761fe3,1427
3,outfit.feaa16af0f6b4a96b65f8a29ace77979,Tahiti Earrings Green/Gold,The Tahiti Earrings are the perfect statement ...,group.d940849fd6a77efb5fd69121e43bb3ad,o_00037,2019-03-22 11:34:50.000,1600.0,590.0,1180.0,"[Multi Season, Jewelry, Formal, Green, Metalli...","[Seasons, Category, Occasion, Color, Details, ...",outfit.feaa16af0f6b4a96b65f8a29ace77979,814
4,outfit.fea3bb7d8ff54872ad84977465e29da4,Asbjorg Aqua Haze Leopard Skirt,Own this look for a vintage and everyday look....,group.fc687f43497199532a11b0f991ebee04,o_00530,2019-05-21 14:33:03.000,1700.0,590.0,1180.0,"[Everyday, Midi, Ruffles, Wrap, Skirts, Women,...","[Occasion, Length, Details, Fit, Category, Gen...",outfit.fea3bb7d8ff54872ad84977465e29da4,572


In [134]:
merged_df.iloc[0]

id                            outfit.ffd83466cdb84a0dba02339aa0c72f73
name                                     Mariposa Earrings AB-Crystal
description         The Mariposa Earrings are the perfect statemen...
group                          group.4f70fa1707b559c0db341fa53997e52f
owner                                                         o_00037
timeCreated                                   2019-03-22 11:52:11.000
retailPrice                                                    1600.0
pricePerWeek                                                    250.0
pricePerMonth                                                   500.0
outfit_tags         [Jewelry, Cecilie Melli, Statement, Metallic, ...
tag_categories         [Category, Brand, Occasion, Details, Occasion]
item_id_original              outfit.ffd83466cdb84a0dba02339aa0c72f73
item_id                                                           482
Name: 0, dtype: object

## Xử lý feature3 ( embedings đặc trưng hình ảnh )

In [135]:
pictures.info()

<class 'pandas.DataFrame'>
RangeIndex: 2223 entries, 0 to 2222
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   picture.id    2223 non-null   str  
 1   outfit.id     2223 non-null   str  
 2   displayOrder  2223 non-null   int64
 3   file_name     2223 non-null   str  
dtypes: int64(1), str(3)
memory usage: 69.6 KB


In [136]:
pictures.head()

,picture.id,outfit.id,displayOrder,file_name
0,picture.0000cdba64314d84a49ed1c266589cc0,outfit.794483397da8425a813301eecf9828c6,0,0000cdba64314d84a49ed1c266589cc0.jpg
1,picture.0010c2e161154d6893734981d5455e76,outfit.9387d05b47f906c5,0,0010c2e161154d6893734981d5455e76.jpg
2,picture.0013342f663843d4b7397b4b88b98e6d,outfit.b24fc57f02ac4e4c9061761ca18d9d3c,0,0013342f663843d4b7397b4b88b98e6d.jpg
3,picture.001a58ea68da426384567b8cccc0c8a6,outfit.e989b8cb4a814d97b642e1cb326f47e6,0,001a58ea68da426384567b8cccc0c8a6.jpg
4,picture.0021a36dd3b04f7ebff9430b7801d44d,outfit.070bc7bd72154b5a805f52f086fb72ed,0,0021a36dd3b04f7ebff9430b7801d44d.jpg


In [137]:
pictures['outfit.id'].nunique()

2223

In [138]:
print(pictures['outfit.id'].isna().sum())

0


In [139]:
import pandas as pd

# Giả định df là DataFrame của bạn
grouped_sorted_df = (
    pictures.sort_values(by=['outfit.id', 'displayOrder'])  # Sắp xếp theo outfit.id và displayOrder
      .groupby('outfit.id', as_index=False)
      .agg({
          'picture.id': lambda x: list(x),
          'file_name': lambda x: list(x),
          'displayOrder': lambda x: list(x)
      })
)



In [140]:
grouped_sorted_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2223 entries, 0 to 2222
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   outfit.id     2223 non-null   str   
 1   picture.id    2223 non-null   object
 2   file_name     2223 non-null   object
 3   displayOrder  2223 non-null   object
dtypes: object(3), str(1)
memory usage: 69.6+ KB


In [141]:
grouped_sorted_df.head()

,outfit.id,picture.id,file_name,displayOrder
0,outfit.005031a3c94b4073b25088b0e19f0e99,[picture.3c1352a3e42d4c9e920edf8eb969ed40],[3c1352a3e42d4c9e920edf8eb969ed40.jpg],[0]
1,outfit.005123277cc440ffbac2d52252cfcac1,[picture.9c39d325e1b34a80b5f1a9b734c04129],[9c39d325e1b34a80b5f1a9b734c04129.jpg],[0]
2,outfit.006b61bc5e404f37a41b85e6ae915723,[picture.29739c734ae14c73bae71a7303f2a46d],[29739c734ae14c73bae71a7303f2a46d.jpg],[0]
3,outfit.007a4828b12b40ed9ed3db0c5803b813,[picture.724d682edbd14a0b9edf9245d5a2e137],[724d682edbd14a0b9edf9245d5a2e137.jpg],[0]
4,outfit.00c68b581a95458991a876f791e739ab,[picture.27a00e5cb9ad4c5ea953b9c9e4abf8a9],[27a00e5cb9ad4c5ea953b9c9e4abf8a9.jpg],[0]


In [142]:
grouped_sorted_df['outfit.id'].nunique()

2223

In [143]:
grouped_sorted_df.iloc[2000]

outfit.id          outfit.e0f170b8b51f4ae4a7e00047b9ea8d79
picture.id      [picture.5edd410bd98c4dcabf86f94e359a6319]
file_name           [5edd410bd98c4dcabf86f94e359a6319.jpg]
displayOrder                                           [0]
Name: 2000, dtype: object

In [144]:
grouped_sorted_df.iloc[2000]['picture.id']

['picture.5edd410bd98c4dcabf86f94e359a6319']

In [145]:
grouped_sorted_df.iloc[2000]['picture.id'][0]

'picture.5edd410bd98c4dcabf86f94e359a6319'

In [146]:
print(merged_df.isna().sum())

id                  0
name                0
description         0
group               0
owner               0
timeCreated         0
retailPrice         3
pricePerWeek        0
pricePerMonth       0
outfit_tags         0
tag_categories      0
item_id_original    0
item_id             0
dtype: int64


In [147]:
print(grouped_sorted_df.isna().sum())

outfit.id       0
picture.id      0
file_name       0
displayOrder    0
dtype: int64


In [148]:
# Kiểm tra các giá trị trong 'id' của merged_df không tồn tại trong 'outfit.id' của grouped_sorted_df
non_matching_ids = merged_df[~merged_df['id'].isin(grouped_sorted_df['outfit.id'])]

# Hiển thị các giá trị không khớp
print(non_matching_ids[['id']])

Empty DataFrame
Columns: [id]
Index: []


In [149]:
import pandas as pd

# Giả định df_a và df_b đã được đọc vào
items_features_df = pd.merge(
    merged_df, grouped_sorted_df,
    left_on='item_id_original',     # Cột tham chiếu từ df_a
    right_on='outfit.id',  # Cột tham chiếu từ df_b
    how='inner'       # Merge theo cách "inner" để giữ lại các hàng khớp
)




In [150]:
items_features_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2194 entries, 0 to 2193
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                2194 non-null   str    
 1   name              2194 non-null   str    
 2   description       2194 non-null   str    
 3   group             2194 non-null   str    
 4   owner             2194 non-null   str    
 5   timeCreated       2194 non-null   str    
 6   retailPrice       2191 non-null   float64
 7   pricePerWeek      2194 non-null   float64
 8   pricePerMonth     2194 non-null   float64
 9   outfit_tags       2194 non-null   object 
 10  tag_categories    2194 non-null   object 
 11  item_id_original  2194 non-null   str    
 12  item_id           2194 non-null   int64  
 13  outfit.id         2194 non-null   str    
 14  picture.id        2194 non-null   object 
 15  file_name         2194 non-null   object 
 16  displayOrder      2194 non-null   object 
dtypes: flo

In [151]:
print(items_features_df.isna().sum())

id                  0
name                0
description         0
group               0
owner               0
timeCreated         0
retailPrice         3
pricePerWeek        0
pricePerMonth       0
outfit_tags         0
tag_categories      0
item_id_original    0
item_id             0
outfit.id           0
picture.id          0
file_name           0
displayOrder        0
dtype: int64


In [152]:
items_features_df['outfit.id'].nunique()

2194

In [153]:
items_features_df['id'].nunique()

2194

In [154]:
items_features_df.head()

,id,name,description,group,owner,timeCreated,retailPrice,pricePerWeek,pricePerMonth,outfit_tags,tag_categories,item_id_original,item_id,outfit.id,picture.id,file_name,displayOrder
0,outfit.ffd83466cdb84a0dba02339aa0c72f73,Mariposa Earrings AB-Crystal,The Mariposa Earrings are the perfect statemen...,group.4f70fa1707b559c0db341fa53997e52f,o_00037,2019-03-22 11:52:11.000,1600.0,250.0,500.0,"[Jewelry, Cecilie Melli, Statement, Metallic, ...","[Category, Brand, Occasion, Details, Occasion]",outfit.ffd83466cdb84a0dba02339aa0c72f73,482,outfit.ffd83466cdb84a0dba02339aa0c72f73,[picture.aad5bad820ae40e2af66dda30b887fd3],[aad5bad820ae40e2af66dda30b887fd3.jpg],[0]
1,outfit.ff08dd87bf144defaa332eac0bf5bd4e,The Gisele Dress,This mini satin dress is made in a beautiful o...,group.a65e95a2f1823633a8d0b07b7fff2e7b,o_00740,2020-02-27 12:51:07.073,2500.0,750.0,1500.0,"[Statement, S, Bastet Noir, Mini, Dresses, Wom...","[Occasion, Size, Brand, Length, Category, Gend...",outfit.ff08dd87bf144defaa332eac0bf5bd4e,580,outfit.ff08dd87bf144defaa332eac0bf5bd4e,[picture.1adaeb284b764f3fb0b553e4df7067d3],[1adaeb284b764f3fb0b553e4df7067d3.jpg],[0]
2,outfit.fec66293d37940b79dde2c1acd761fe3,Evie Deauville Mauve Long Sleeve Dress,The Evie Long Sleeve Dress is made in a flowy ...,group.fcc139e28abeaf451c19d46bf311fcb0,o_00530,2019-09-23 09:47:20.000,2200.0,660.0,1320.0,"[Ruffles, Women, Midi, Purple, Multi Season, S...","[Details, Gender, Length, Color, Seasons, Mate...",outfit.fec66293d37940b79dde2c1acd761fe3,1427,outfit.fec66293d37940b79dde2c1acd761fe3,[picture.1854c475fbfa402f9867c970a71f4656],[1854c475fbfa402f9867c970a71f4656.jpg],[0]
3,outfit.feaa16af0f6b4a96b65f8a29ace77979,Tahiti Earrings Green/Gold,The Tahiti Earrings are the perfect statement ...,group.d940849fd6a77efb5fd69121e43bb3ad,o_00037,2019-03-22 11:34:50.000,1600.0,590.0,1180.0,"[Multi Season, Jewelry, Formal, Green, Metalli...","[Seasons, Category, Occasion, Color, Details, ...",outfit.feaa16af0f6b4a96b65f8a29ace77979,814,outfit.feaa16af0f6b4a96b65f8a29ace77979,[picture.7e2df174e9d84919bea360f6da43e59a],[7e2df174e9d84919bea360f6da43e59a.jpg],[0]
4,outfit.fea3bb7d8ff54872ad84977465e29da4,Asbjorg Aqua Haze Leopard Skirt,Own this look for a vintage and everyday look....,group.fc687f43497199532a11b0f991ebee04,o_00530,2019-05-21 14:33:03.000,1700.0,590.0,1180.0,"[Everyday, Midi, Ruffles, Wrap, Skirts, Women,...","[Occasion, Length, Details, Fit, Category, Gen...",outfit.fea3bb7d8ff54872ad84977465e29da4,572,outfit.fea3bb7d8ff54872ad84977465e29da4,[picture.376103f9513c4440a29824e469e330f8],[376103f9513c4440a29824e469e330f8.jpg],[0]


In [155]:
items_features_df.iloc[0]

id                            outfit.ffd83466cdb84a0dba02339aa0c72f73
name                                     Mariposa Earrings AB-Crystal
description         The Mariposa Earrings are the perfect statemen...
group                          group.4f70fa1707b559c0db341fa53997e52f
owner                                                         o_00037
timeCreated                                   2019-03-22 11:52:11.000
retailPrice                                                    1600.0
pricePerWeek                                                    250.0
pricePerMonth                                                   500.0
outfit_tags         [Jewelry, Cecilie Melli, Statement, Metallic, ...
tag_categories         [Category, Brand, Occasion, Details, Occasion]
item_id_original              outfit.ffd83466cdb84a0dba02339aa0c72f73
item_id                                                           482
outfit.id                     outfit.ffd83466cdb84a0dba02339aa0c72f73
picture.id          

In [339]:
# # Thực hiện merge với how='left'
# left_merged_df = pd.merge(
#     merged_df, grouped_sorted_df,
#     left_on='id',     # Cột tham chiếu từ df_a
#     right_on='outfit.id',  # Cột tham chiếu từ df_b
#     how='left'       # Merge theo cách "left" để giữ lại tất cả các hàng từ merged_df
# )

# # Lọc ra các hàng không khớp
# non_matching_rows = left_merged_df[left_merged_df['outfit.id'].isna()]

# # Hiển thị các hàng không khớp
# non_matching_rows.head()

In [340]:
# left_merged_df.info()

## Tạo image_list

In [156]:
image_list_df = items_features_df[['item_id_original' ,'item_id', 'file_name']]

In [157]:
image_list_df.head()

,item_id_original,item_id,file_name
0,outfit.ffd83466cdb84a0dba02339aa0c72f73,482,[aad5bad820ae40e2af66dda30b887fd3.jpg]
1,outfit.ff08dd87bf144defaa332eac0bf5bd4e,580,[1adaeb284b764f3fb0b553e4df7067d3.jpg]
2,outfit.fec66293d37940b79dde2c1acd761fe3,1427,[1854c475fbfa402f9867c970a71f4656.jpg]
3,outfit.feaa16af0f6b4a96b65f8a29ace77979,814,[7e2df174e9d84919bea360f6da43e59a.jpg]
4,outfit.fea3bb7d8ff54872ad84977465e29da4,572,[376103f9513c4440a29824e469e330f8.jpg]


In [158]:
image_list_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2194 entries, 0 to 2193
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   item_id_original  2194 non-null   str   
 1   item_id           2194 non-null   int64 
 2   file_name         2194 non-null   object
dtypes: int64(1), object(1), str(1)
memory usage: 51.6+ KB


In [159]:
# Sắp xếp theo giá trị user_id tăng dần
image_list_df = image_list_df.sort_values(by='item_id')

image_list_df.reset_index(drop=True, inplace=True)  # Đặt lại chỉ mục

In [160]:
image_list_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2194 entries, 0 to 2193
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   item_id_original  2194 non-null   str   
 1   item_id           2194 non-null   int64 
 2   file_name         2194 non-null   object
dtypes: int64(1), object(1), str(1)
memory usage: 51.6+ KB


In [161]:
image_list_df.head()

,item_id_original,item_id,file_name
0,outfit.7cc6bd8e645a41d2837976a0d09e3ca9,0,[7ff7061578c34d1bab1a6ea843b1ffa9.jpg]
1,outfit.7ee7542297f944ecb49f4fe5b44728f7,1,[0b128533a20941d3b4483ec55b94f3c8.jpg]
2,outfit.e8e4d130fa3e4f108127dc74fd854a75,2,[2745413be437408b943701d3b1df26ba.jpg]
3,outfit.7395b78d10084aebb906f8f83bfd89cc,3,[1b7695b2c0824da5b46d0e2f74b6f0ed.jpg]
4,outfit.c929e1bd6aa940f7a34c16471389c460,4,[09326e8ed84b4a1fae03a7911a2ca901.jpg]


In [162]:
# Ghi vào file user_list.txt, không bao gồm dòng tên cột
image_list_df.to_csv(rf"{ROOT}\\image_list.txt", sep=' ', index=False, header=False)

print("File image_list.txt đã được tạo.")

File image_list.txt đã được tạo.


## Tạo file items_features 

In [163]:
# Tạo cột item_id từ cột index trong df_b
fe_df = items_features_df.copy()

fe_df.head()

,id,name,description,group,owner,timeCreated,retailPrice,pricePerWeek,pricePerMonth,outfit_tags,tag_categories,item_id_original,item_id,outfit.id,picture.id,file_name,displayOrder
0,outfit.ffd83466cdb84a0dba02339aa0c72f73,Mariposa Earrings AB-Crystal,The Mariposa Earrings are the perfect statemen...,group.4f70fa1707b559c0db341fa53997e52f,o_00037,2019-03-22 11:52:11.000,1600.0,250.0,500.0,"[Jewelry, Cecilie Melli, Statement, Metallic, ...","[Category, Brand, Occasion, Details, Occasion]",outfit.ffd83466cdb84a0dba02339aa0c72f73,482,outfit.ffd83466cdb84a0dba02339aa0c72f73,[picture.aad5bad820ae40e2af66dda30b887fd3],[aad5bad820ae40e2af66dda30b887fd3.jpg],[0]
1,outfit.ff08dd87bf144defaa332eac0bf5bd4e,The Gisele Dress,This mini satin dress is made in a beautiful o...,group.a65e95a2f1823633a8d0b07b7fff2e7b,o_00740,2020-02-27 12:51:07.073,2500.0,750.0,1500.0,"[Statement, S, Bastet Noir, Mini, Dresses, Wom...","[Occasion, Size, Brand, Length, Category, Gend...",outfit.ff08dd87bf144defaa332eac0bf5bd4e,580,outfit.ff08dd87bf144defaa332eac0bf5bd4e,[picture.1adaeb284b764f3fb0b553e4df7067d3],[1adaeb284b764f3fb0b553e4df7067d3.jpg],[0]
2,outfit.fec66293d37940b79dde2c1acd761fe3,Evie Deauville Mauve Long Sleeve Dress,The Evie Long Sleeve Dress is made in a flowy ...,group.fcc139e28abeaf451c19d46bf311fcb0,o_00530,2019-09-23 09:47:20.000,2200.0,660.0,1320.0,"[Ruffles, Women, Midi, Purple, Multi Season, S...","[Details, Gender, Length, Color, Seasons, Mate...",outfit.fec66293d37940b79dde2c1acd761fe3,1427,outfit.fec66293d37940b79dde2c1acd761fe3,[picture.1854c475fbfa402f9867c970a71f4656],[1854c475fbfa402f9867c970a71f4656.jpg],[0]
3,outfit.feaa16af0f6b4a96b65f8a29ace77979,Tahiti Earrings Green/Gold,The Tahiti Earrings are the perfect statement ...,group.d940849fd6a77efb5fd69121e43bb3ad,o_00037,2019-03-22 11:34:50.000,1600.0,590.0,1180.0,"[Multi Season, Jewelry, Formal, Green, Metalli...","[Seasons, Category, Occasion, Color, Details, ...",outfit.feaa16af0f6b4a96b65f8a29ace77979,814,outfit.feaa16af0f6b4a96b65f8a29ace77979,[picture.7e2df174e9d84919bea360f6da43e59a],[7e2df174e9d84919bea360f6da43e59a.jpg],[0]
4,outfit.fea3bb7d8ff54872ad84977465e29da4,Asbjorg Aqua Haze Leopard Skirt,Own this look for a vintage and everyday look....,group.fc687f43497199532a11b0f991ebee04,o_00530,2019-05-21 14:33:03.000,1700.0,590.0,1180.0,"[Everyday, Midi, Ruffles, Wrap, Skirts, Women,...","[Occasion, Length, Details, Fit, Category, Gen...",outfit.fea3bb7d8ff54872ad84977465e29da4,572,outfit.fea3bb7d8ff54872ad84977465e29da4,[picture.376103f9513c4440a29824e469e330f8],[376103f9513c4440a29824e469e330f8.jpg],[0]


## => Tạo feature1 và feature2

In [164]:
# Chuyển 'tag_categories' thành chuỗi, nối các phần tử trong danh sách với khoảng trắng
fe_df['outfit_tags'] = fe_df['outfit_tags'].apply(lambda x: ' '.join(x) if isinstance(x, list) else x)

# Tạo cột feature1: kết hợp name và tag_categories với dấu cách giữa chúng
fe_df['feature1'] = fe_df['name'] + ' ' + fe_df['outfit_tags']

# Tạo cột feature2: là cột description từ df_a
fe_df['feature2'] = fe_df['description']

In [165]:
fe_df.head()

,id,name,description,group,owner,timeCreated,retailPrice,pricePerWeek,pricePerMonth,outfit_tags,tag_categories,item_id_original,item_id,outfit.id,picture.id,file_name,displayOrder,feature1,feature2
0,outfit.ffd83466cdb84a0dba02339aa0c72f73,Mariposa Earrings AB-Crystal,The Mariposa Earrings are the perfect statemen...,group.4f70fa1707b559c0db341fa53997e52f,o_00037,2019-03-22 11:52:11.000,1600.0,250.0,500.0,Jewelry Cecilie Melli Statement Metallic Formal,"[Category, Brand, Occasion, Details, Occasion]",outfit.ffd83466cdb84a0dba02339aa0c72f73,482,outfit.ffd83466cdb84a0dba02339aa0c72f73,[picture.aad5bad820ae40e2af66dda30b887fd3],[aad5bad820ae40e2af66dda30b887fd3.jpg],[0],Mariposa Earrings AB-Crystal Jewelry Cecilie M...,The Mariposa Earrings are the perfect statemen...
1,outfit.ff08dd87bf144defaa332eac0bf5bd4e,The Gisele Dress,This mini satin dress is made in a beautiful o...,group.a65e95a2f1823633a8d0b07b7fff2e7b,o_00740,2020-02-27 12:51:07.073,2500.0,750.0,1500.0,Statement S Bastet Noir Mini Dresses Women Gre...,"[Occasion, Size, Brand, Length, Category, Gend...",outfit.ff08dd87bf144defaa332eac0bf5bd4e,580,outfit.ff08dd87bf144defaa332eac0bf5bd4e,[picture.1adaeb284b764f3fb0b553e4df7067d3],[1adaeb284b764f3fb0b553e4df7067d3.jpg],[0],The Gisele Dress Statement S Bastet Noir Mini ...,This mini satin dress is made in a beautiful o...
2,outfit.fec66293d37940b79dde2c1acd761fe3,Evie Deauville Mauve Long Sleeve Dress,The Evie Long Sleeve Dress is made in a flowy ...,group.fcc139e28abeaf451c19d46bf311fcb0,o_00530,2019-09-23 09:47:20.000,2200.0,660.0,1320.0,Ruffles Women Midi Purple Multi Season Synthet...,"[Details, Gender, Length, Color, Seasons, Mate...",outfit.fec66293d37940b79dde2c1acd761fe3,1427,outfit.fec66293d37940b79dde2c1acd761fe3,[picture.1854c475fbfa402f9867c970a71f4656],[1854c475fbfa402f9867c970a71f4656.jpg],[0],Evie Deauville Mauve Long Sleeve Dress Ruffles...,The Evie Long Sleeve Dress is made in a flowy ...
3,outfit.feaa16af0f6b4a96b65f8a29ace77979,Tahiti Earrings Green/Gold,The Tahiti Earrings are the perfect statement ...,group.d940849fd6a77efb5fd69121e43bb3ad,o_00037,2019-03-22 11:34:50.000,1600.0,590.0,1180.0,Multi Season Jewelry Formal Green Metallic Wom...,"[Seasons, Category, Occasion, Color, Details, ...",outfit.feaa16af0f6b4a96b65f8a29ace77979,814,outfit.feaa16af0f6b4a96b65f8a29ace77979,[picture.7e2df174e9d84919bea360f6da43e59a],[7e2df174e9d84919bea360f6da43e59a.jpg],[0],Tahiti Earrings Green/Gold Multi Season Jewelr...,The Tahiti Earrings are the perfect statement ...
4,outfit.fea3bb7d8ff54872ad84977465e29da4,Asbjorg Aqua Haze Leopard Skirt,Own this look for a vintage and everyday look....,group.fc687f43497199532a11b0f991ebee04,o_00530,2019-05-21 14:33:03.000,1700.0,590.0,1180.0,Everyday Midi Ruffles Wrap Skirts Women S Visc...,"[Occasion, Length, Details, Fit, Category, Gen...",outfit.fea3bb7d8ff54872ad84977465e29da4,572,outfit.fea3bb7d8ff54872ad84977465e29da4,[picture.376103f9513c4440a29824e469e330f8],[376103f9513c4440a29824e469e330f8.jpg],[0],Asbjorg Aqua Haze Leopard Skirt Everyday Midi ...,Own this look for a vintage and everyday look....


In [166]:
fe_df.iloc[0]

id                            outfit.ffd83466cdb84a0dba02339aa0c72f73
name                                     Mariposa Earrings AB-Crystal
description         The Mariposa Earrings are the perfect statemen...
group                          group.4f70fa1707b559c0db341fa53997e52f
owner                                                         o_00037
timeCreated                                   2019-03-22 11:52:11.000
retailPrice                                                    1600.0
pricePerWeek                                                    250.0
pricePerMonth                                                   500.0
outfit_tags           Jewelry Cecilie Melli Statement Metallic Formal
tag_categories         [Category, Brand, Occasion, Details, Occasion]
item_id_original              outfit.ffd83466cdb84a0dba02339aa0c72f73
item_id                                                           482
outfit.id                     outfit.ffd83466cdb84a0dba02339aa0c72f73
picture.id          

=> path sẵn có từ dữ liệu gốc ( chưa detect phần thừa )

In [352]:
# import numpy as np
# import os
# # Tạo cột feature3: là danh sách embedings từ df_a

# # Thư mục chứa các file embeddings
# embeddings_dir = '/kaggle/input/vibrent-clothes-rental-dataset/embeddings/EfficientNet_V2_L_final'

# # Hàm tạo danh sách đường dẫn đầy đủ đến tệp .npy
# def create_embedding_paths(picture_ids):
#     return [os.path.join(embeddings_dir, f"{pic_id}.npy") for pic_id in picture_ids]

# # Sử dụng function
# fe_df['embedding_paths'] = fe_df['picture.id'].apply(create_embedding_paths)

=> path mới (đã detect phần thừa lưu ý số chiều )

In [167]:
import numpy as np
import os

# Thư mục chứa các file embeddings
embeddings_dir = r'E:\\DoCode\\CD2\\source\\Source\\get_hrs_rs\\rs\\get10k_data\\output_10k_sample\\embeddings_10k'

# Hàm tạo danh sách đường dẫn đầy đủ đến tệp .npy
def create_embedding_paths(picture_ids):
    return [os.path.join(embeddings_dir, f"{pic_id.split('.')[1]}.npy") for pic_id in picture_ids]

# Sử dụng function
fe_df['embedding_paths'] = fe_df['picture.id'].apply(create_embedding_paths)

## combine feature3 embeddings#1 ( mean )

In [168]:
cbf_f31 = fe_df.copy()

In [169]:
cbf_f31.iloc[0]

id                            outfit.ffd83466cdb84a0dba02339aa0c72f73
name                                     Mariposa Earrings AB-Crystal
description         The Mariposa Earrings are the perfect statemen...
group                          group.4f70fa1707b559c0db341fa53997e52f
owner                                                         o_00037
timeCreated                                   2019-03-22 11:52:11.000
retailPrice                                                    1600.0
pricePerWeek                                                    250.0
pricePerMonth                                                   500.0
outfit_tags           Jewelry Cecilie Melli Statement Metallic Formal
tag_categories         [Category, Brand, Occasion, Details, Occasion]
item_id_original              outfit.ffd83466cdb84a0dba02339aa0c72f73
item_id                                                           482
outfit.id                     outfit.ffd83466cdb84a0dba02339aa0c72f73
picture.id          

In [170]:
cbf_f31.iloc[0]['embedding_paths']

['E:\\\\DoCode\\\\CD2\\\\source\\\\Source\\\\get_hrs_rs\\\\rs\\\\get10k_data\\\\output_10k_sample\\\\embeddings_10k\\aad5bad820ae40e2af66dda30b887fd3.npy']

In [171]:

def load_mean_embeddings(paths):
    """
    Load và combine embeddings của các ảnh trong một outfit
    paths: list các đường dẫn đến embedding files của một outfit
    """
    outfit_embeddings = []
    
    for path in paths:
        if os.path.isfile(path):
            # Load embedding và giữ nguyên shape
            embedding = np.load(path)
            # Print để debug
            # print("Single image embedding shape:", embedding.shape)  # Should be (1280,)
            outfit_embeddings.append(embedding.astype(np.float32))
    
    if outfit_embeddings:
        # Đảm bảo mỗi embedding là 1D array
        outfit_embeddings = [emb.flatten() for emb in outfit_embeddings]
        # Combine embeddings
        combined_embedding = np.mean(outfit_embeddings, axis=0)
        # Print để debug
        # print("Combined embedding shape:", combined_embedding.shape)  # Should be (1280,)
        return combined_embedding.tolist()
    return [0] * 1280  # Return zero vector nếu không có embedding


# Sử dụng function
cbf_f31['feature3'] = cbf_f31['embedding_paths'].apply(load_mean_embeddings)

# Kiểm tra sau khi load
print("Sample of feature3 first element:", len(cbf_f31['feature3'].iloc[0]))  # Should be 1280

Sample of feature3 first element: 1280


## combine feature3 embeddings#2 ( weighted )

In [172]:
cbf_f32 = fe_df.copy()

In [173]:
cbf_f32.iloc[0]

id                            outfit.ffd83466cdb84a0dba02339aa0c72f73
name                                     Mariposa Earrings AB-Crystal
description         The Mariposa Earrings are the perfect statemen...
group                          group.4f70fa1707b559c0db341fa53997e52f
owner                                                         o_00037
timeCreated                                   2019-03-22 11:52:11.000
retailPrice                                                    1600.0
pricePerWeek                                                    250.0
pricePerMonth                                                   500.0
outfit_tags           Jewelry Cecilie Melli Statement Metallic Formal
tag_categories         [Category, Brand, Occasion, Details, Occasion]
item_id_original              outfit.ffd83466cdb84a0dba02339aa0c72f73
item_id                                                           482
outfit.id                     outfit.ffd83466cdb84a0dba02339aa0c72f73
picture.id          

In [174]:
cbf_f32.iloc[0]['embedding_paths']

['E:\\\\DoCode\\\\CD2\\\\source\\\\Source\\\\get_hrs_rs\\\\rs\\\\get10k_data\\\\output_10k_sample\\\\embeddings_10k\\aad5bad820ae40e2af66dda30b887fd3.npy']

In [175]:

def load_weighted_embeddings(paths, display_orders):
    """
    Load và combine embeddings của các ảnh trong một outfit với trọng số dựa trên display order.
    paths: list các đường dẫn đến embedding files của một outfit.
    display_orders: list thứ tự ưu tiên của các ảnh trong outfit.
    """
    outfit_embeddings = []
    valid_orders = []

    for idx, path in enumerate(paths):
        if os.path.isfile(path):
            embedding = np.load(path).flatten().astype(np.float32)
            outfit_embeddings.append(embedding)
            valid_orders.append(display_orders[idx])

    if outfit_embeddings:
        # Tính trọng số dựa vào displayOrder
        weights = 1 / (np.array(valid_orders) + 1)
        weights = weights / np.sum(weights)  # Chuẩn hóa trọng số
        
        # Áp dụng trọng số vào embeddings
        weighted_embedding = np.sum([w * emb for w, emb in zip(weights, outfit_embeddings)], axis=0)
        
        return weighted_embedding.tolist()

    # Trả về vector zero nếu không có embedding hợp lệ
    return [0] * 1280

# Áp dụng cho DataFrame fe_df
cbf_f32['feature3'] = cbf_f32.apply(
    lambda row: load_weighted_embeddings(row['embedding_paths'], row['displayOrder']),
    axis=1
)

# Kiểm tra kết quả
print("Feature3 sample embedding length:", len(cbf_f32['feature3'].iloc[0]))  # Should be 1280


Feature3 sample embedding length: 1280


## combine feature3 embeddings#3 ( max )

In [176]:
cbf_f33 = fe_df.copy()

In [177]:
cbf_f33.iloc[0]

id                            outfit.ffd83466cdb84a0dba02339aa0c72f73
name                                     Mariposa Earrings AB-Crystal
description         The Mariposa Earrings are the perfect statemen...
group                          group.4f70fa1707b559c0db341fa53997e52f
owner                                                         o_00037
timeCreated                                   2019-03-22 11:52:11.000
retailPrice                                                    1600.0
pricePerWeek                                                    250.0
pricePerMonth                                                   500.0
outfit_tags           Jewelry Cecilie Melli Statement Metallic Formal
tag_categories         [Category, Brand, Occasion, Details, Occasion]
item_id_original              outfit.ffd83466cdb84a0dba02339aa0c72f73
item_id                                                           482
outfit.id                     outfit.ffd83466cdb84a0dba02339aa0c72f73
picture.id          

In [178]:
cbf_f33.iloc[0]['embedding_paths']

['E:\\\\DoCode\\\\CD2\\\\source\\\\Source\\\\get_hrs_rs\\\\rs\\\\get10k_data\\\\output_10k_sample\\\\embeddings_10k\\aad5bad820ae40e2af66dda30b887fd3.npy']

In [179]:

def load_max_embeddings(paths):
    """
    Load và combine embeddings của các ảnh trong một outfit
    paths: list các đường dẫn đến embedding files của một outfit
    """
    outfit_embeddings = []
    
    for path in paths:
        if os.path.isfile(path):
            # Load embedding và giữ nguyên shape
            embedding = np.load(path)
            # Print để debug
            # print("Single image embedding shape:", embedding.shape)  # Should be (1280,)
            outfit_embeddings.append(embedding.astype(np.float32))
    
    if outfit_embeddings:
        # Đảm bảo mỗi embedding là 1D array
        outfit_embeddings = [emb.flatten() for emb in outfit_embeddings]
        # Combine embeddings
        combined_embedding = np.max(outfit_embeddings, axis=0)
        # Print để debug
        # print("Combined embedding shape:", combined_embedding.shape)  # Should be (1280,)
        return combined_embedding.tolist()
    return [0] * 1280  # Return zero vector nếu không có embedding


# Sử dụng function
cbf_f33['feature3'] = cbf_f33['embedding_paths'].apply(load_max_embeddings)

# Kiểm tra sau khi load
print("Sample of feature3 first element:", len(cbf_f33['feature3'].iloc[0]))  # Should be 1280

Sample of feature3 first element: 1280


## Dimensionality reduction with PCA

In [180]:
cbf_f31['feature3'].head()

0    [0.0, 2.368340492248535, 0.008222006261348724,...
1    [0.0, 0.017007915303111076, 0.0, 0.37110745906...
2    [0.07040582597255707, 0.19717185199260712, 0.0...
3    [0.028707018122076988, 2.110269069671631, 0.02...
4    [0.010761134326457977, 0.0, 0.4700208902359009...
Name: feature3, dtype: object

In [181]:
cbf_f32['feature3'].head()

0    [0.0, 2.368340492248535, 0.008222006261348724,...
1    [0.0, 0.017007915303111076, 0.0, 0.37110745906...
2    [0.07040582597255707, 0.19717185199260712, 0.0...
3    [0.028707018122076988, 2.110269069671631, 0.02...
4    [0.010761134326457977, 0.0, 0.4700208902359009...
Name: feature3, dtype: object

In [182]:
cbf_f33['feature3'].head()

0    [0.0, 2.368340492248535, 0.008222006261348724,...
1    [0.0, 0.017007915303111076, 0.0, 0.37110745906...
2    [0.07040582597255707, 0.19717185199260712, 0.0...
3    [0.028707018122076988, 2.110269069671631, 0.02...
4    [0.010761134326457977, 0.0, 0.4700208902359009...
Name: feature3, dtype: object

In [183]:
# Gán lại kết quả vào fe_df sau khi kiểm tra
#mean
fe_df['feature3'] = cbf_f31['feature3']
#weight
# fe_df['feature3'] = cbf_f32['feature3']
#max
# fe_df['feature3'] = cbf_f33['feature3']

# Kiểm tra sau khi load
print("Sample of feature3 first element:", len(fe_df['feature3'].iloc[0]))  # Should be 1280

Sample of feature3 first element: 1280


In [184]:
fe_df.iloc[0]

id                            outfit.ffd83466cdb84a0dba02339aa0c72f73
name                                     Mariposa Earrings AB-Crystal
description         The Mariposa Earrings are the perfect statemen...
group                          group.4f70fa1707b559c0db341fa53997e52f
owner                                                         o_00037
timeCreated                                   2019-03-22 11:52:11.000
retailPrice                                                    1600.0
pricePerWeek                                                    250.0
pricePerMonth                                                   500.0
outfit_tags           Jewelry Cecilie Melli Statement Metallic Formal
tag_categories         [Category, Brand, Occasion, Details, Occasion]
item_id_original              outfit.ffd83466cdb84a0dba02339aa0c72f73
item_id                                                           482
outfit.id                     outfit.ffd83466cdb84a0dba02339aa0c72f73
picture.id          

In [185]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
import numpy as np

# Kiểm tra và thay thế NaN bằng vector zero có cùng số chiều
image_embeddings = np.vstack([
    np.array(emb).reshape(1, -1) if isinstance(emb, list) else emb.reshape(1, -1)
    for emb in fe_df['feature3']
])
print("Image embeddings shape before normalization:", image_embeddings.shape)

# Thay thế NaN bằng vector zero
image_embeddings = np.nan_to_num(image_embeddings)

# Chuẩn hóa embeddings trước khi áp dụng PCA
image_embeddings_normalized = normalize(image_embeddings)

# Xác định số chiều mong muốn cho PCA
n_samples, n_features = image_embeddings_normalized.shape
n_components = min(768, n_features, n_samples)

# Áp dụng PCA
pca = PCA(n_components=n_components)
image_embeddings_reduced = pca.fit_transform(image_embeddings_normalized)

# Cập nhật cột feature3 mà không làm thay đổi thứ tự ban đầu
fe_df['feature3'] = list(image_embeddings_reduced)

# Kiểm tra kết quả
print("Feature3 vector size:", len(fe_df['feature3'].iloc[0]))


Image embeddings shape before normalization: (2194, 1280)
Feature3 vector size: 768


In [186]:
print(len(fe_df['feature3'].iloc[0]))  # Kiểm tra vector sau giảm chiều
print("Tổng số mẫu:", len(fe_df))


768
Tổng số mẫu: 2194


In [187]:
fe_df.iloc[0]

id                            outfit.ffd83466cdb84a0dba02339aa0c72f73
name                                     Mariposa Earrings AB-Crystal
description         The Mariposa Earrings are the perfect statemen...
group                          group.4f70fa1707b559c0db341fa53997e52f
owner                                                         o_00037
timeCreated                                   2019-03-22 11:52:11.000
retailPrice                                                    1600.0
pricePerWeek                                                    250.0
pricePerMonth                                                   500.0
outfit_tags           Jewelry Cecilie Melli Statement Metallic Formal
tag_categories         [Category, Brand, Occasion, Details, Occasion]
item_id_original              outfit.ffd83466cdb84a0dba02339aa0c72f73
item_id                                                           482
outfit.id                     outfit.ffd83466cdb84a0dba02339aa0c72f73
picture.id          

In [188]:
fe_df['displayOrder'].head(10)

0    [0]
1    [0]
2    [0]
3    [0]
4    [0]
5    [0]
6    [0]
7    [0]
8    [0]
9    [0]
Name: displayOrder, dtype: object

In [189]:
fe_df['displayOrder'].tail(10)

2184    [0]
2185    [0]
2186    [0]
2187    [0]
2188    [0]
2189    [0]
2190    [0]
2191    [0]
2192    [0]
2193    [0]
Name: displayOrder, dtype: object

In [190]:
fe_df.iloc[0]['embedding_paths']

['E:\\\\DoCode\\\\CD2\\\\source\\\\Source\\\\get_hrs_rs\\\\rs\\\\get10k_data\\\\output_10k_sample\\\\embeddings_10k\\aad5bad820ae40e2af66dda30b887fd3.npy']

In [191]:
fe_df.iloc[0]['picture.id']

['picture.aad5bad820ae40e2af66dda30b887fd3']

In [192]:
len(fe_df.iloc[0]['feature3'])

768

In [193]:
# Chọn chỉ các cột cần thiết để tạo df_c
df_c = fe_df[['item_id', 'feature1', 'feature2' ,'feature3']]

# Hiển thị kết quả
df_c.head()

,item_id,feature1,feature2,feature3
0,482,Mariposa Earrings AB-Crystal Jewelry Cecilie M...,The Mariposa Earrings are the perfect statemen...,"[-0.1818618411653846, -0.14280182755422194, -0..."
1,580,The Gisele Dress Statement S Bastet Noir Mini ...,This mini satin dress is made in a beautiful o...,"[0.02342565428738875, -0.016955449037710322, 0..."
2,1427,Evie Deauville Mauve Long Sleeve Dress Ruffles...,The Evie Long Sleeve Dress is made in a flowy ...,"[-0.13170364849893063, -0.17881093367455508, -..."
3,814,Tahiti Earrings Green/Gold Multi Season Jewelr...,The Tahiti Earrings are the perfect statement ...,"[-0.2248584545447264, -0.2217712433645414, -0...."
4,572,Asbjorg Aqua Haze Leopard Skirt Everyday Midi ...,Own this look for a vintage and everyday look....,"[0.10333186472271742, 0.04879593792854971, -0...."


In [194]:
df_c['item_id'][0]

482

In [195]:
df_c.iloc[0]

item_id                                                   482
feature1    Mariposa Earrings AB-Crystal Jewelry Cecilie M...
feature2    The Mariposa Earrings are the perfect statemen...
feature3    [-0.1818618411653846, -0.14280182755422194, -0...
Name: 0, dtype: object

In [196]:
df_c['feature1'][0]

'Mariposa Earrings AB-Crystal Jewelry Cecilie Melli Statement Metallic Formal'

In [197]:
df_c['feature2'][0]

'The Mariposa Earrings are the perfect statement jewelry. We recommend wearing your hair up to show them off. '

In [198]:
df_c['feature3'][1200][0]

0.14344012527871722

In [199]:
df_c.info()

<class 'pandas.DataFrame'>
RangeIndex: 2194 entries, 0 to 2193
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   item_id   2194 non-null   int64 
 1   feature1  2194 non-null   str   
 2   feature2  2194 non-null   str   
 3   feature3  2194 non-null   object
dtypes: int64(1), object(1), str(2)
memory usage: 68.7+ KB


In [200]:
df_c.isnull().sum()

item_id     0
feature1    0
feature2    0
feature3    0
dtype: int64

In [201]:
# Lưu DataFrame df_c vào file CSV
df_c.to_csv(rf"{ROOT}\\items_features.csv", index=False)

# Hiển thị thông báo lưu thành công
print("DataFrame đã được lưu vào items_features.csv")


DataFrame đã được lưu vào items_features.csv


# Debug Item feature (sửa khi load items_features trong load_data)

In [202]:
xxx = pd.read_csv(rf"{ROOT}\\items_features.csv")

In [203]:
xxx.info()

<class 'pandas.DataFrame'>
RangeIndex: 2194 entries, 0 to 2193
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   item_id   2194 non-null   int64
 1   feature1  2194 non-null   str  
 2   feature2  2194 non-null   str  
 3   feature3  2194 non-null   str  
dtypes: int64(1), str(3)
memory usage: 68.7 KB


In [204]:
xxx.head()

,item_id,feature1,feature2,feature3
0,482,Mariposa Earrings AB-Crystal Jewelry Cecilie M...,The Mariposa Earrings are the perfect statemen...,[-1.81861841e-01 -1.42801828e-01 -1.73801487e-...
1,580,The Gisele Dress Statement S Bastet Noir Mini ...,This mini satin dress is made in a beautiful o...,[ 2.34256543e-02 -1.69554490e-02 5.82059605e-...
2,1427,Evie Deauville Mauve Long Sleeve Dress Ruffles...,The Evie Long Sleeve Dress is made in a flowy ...,[-1.31703648e-01 -1.78810934e-01 -1.12301135e-...
3,814,Tahiti Earrings Green/Gold Multi Season Jewelr...,The Tahiti Earrings are the perfect statement ...,[-2.24858455e-01 -2.21771243e-01 -1.38866243e-...
4,572,Asbjorg Aqua Haze Leopard Skirt Everyday Midi ...,Own this look for a vintage and everyday look....,[ 1.03331865e-01 4.87959379e-02 -1.19034438e-...


In [205]:
xxx['feature3'][0]

'[-1.81861841e-01 -1.42801828e-01 -1.73801487e-01  2.22816044e-01\n -1.85576959e-01 -8.72849977e-02 -7.47555686e-02  8.47545173e-02\n -2.18261067e-01 -1.26205035e-01  6.62624919e-02 -9.84610015e-02\n -6.17713053e-02  3.41829938e-02  1.95979812e-02 -3.35348070e-02\n  6.84568964e-02 -1.70965173e-01 -6.08901110e-03 -3.71863768e-02\n  2.76397530e-02 -3.35425869e-02 -5.18217055e-02  2.60866879e-02\n -2.95264551e-02  3.93884742e-02 -2.28475665e-02 -6.89586382e-02\n  9.07938965e-02 -7.01272184e-02 -4.31429562e-02 -4.44394261e-02\n -3.99916836e-02 -8.57988959e-02 -4.65823002e-02 -8.67159560e-02\n -5.90027168e-02 -7.47831357e-02 -1.00582427e-01  1.54308927e-01\n -1.92014151e-03  9.11744429e-02  6.61664115e-02  1.91523421e-02\n  1.40991686e-02 -8.40122541e-02  2.08141831e-02  1.13986433e-01\n  3.70342419e-02  3.29378373e-02  6.19243480e-02  2.39760128e-02\n  2.95271117e-03  5.63252171e-02 -1.82790384e-02  4.76904814e-02\n -9.42604760e-02 -3.50648880e-02 -5.34658016e-03  3.61661725e-02\n  6.59257

In [206]:
xxx['feature3'][0][0]

'['

In [207]:
# Chuyển string vector thành numpy array
def parse_vector_string(vector_string):
    # Loại bỏ dấu ngoặc vuông và split theo khoảng trắng
    vector = vector_string.strip('[]').split()
    # Chuyển đổi sang float
    return np.array([float(x) for x in vector])

image_embeddings = np.vstack(
    xxx['feature3'].apply(parse_vector_string).values
)

In [208]:
image_embeddings[0]

array([-1.81861841e-01, -1.42801828e-01, -1.73801487e-01,  2.22816044e-01,
       -1.85576959e-01, -8.72849977e-02, -7.47555686e-02,  8.47545173e-02,
       -2.18261067e-01, -1.26205035e-01,  6.62624919e-02, -9.84610015e-02,
       -6.17713053e-02,  3.41829938e-02,  1.95979812e-02, -3.35348070e-02,
        6.84568964e-02, -1.70965173e-01, -6.08901110e-03, -3.71863768e-02,
        2.76397530e-02, -3.35425869e-02, -5.18217055e-02,  2.60866879e-02,
       -2.95264551e-02,  3.93884742e-02, -2.28475665e-02, -6.89586382e-02,
        9.07938965e-02, -7.01272184e-02, -4.31429562e-02, -4.44394261e-02,
       -3.99916836e-02, -8.57988959e-02, -4.65823002e-02, -8.67159560e-02,
       -5.90027168e-02, -7.47831357e-02, -1.00582427e-01,  1.54308927e-01,
       -1.92014151e-03,  9.11744429e-02,  6.61664115e-02,  1.91523421e-02,
        1.40991686e-02, -8.40122541e-02,  2.08141831e-02,  1.13986433e-01,
        3.70342419e-02,  3.29378373e-02,  6.19243480e-02,  2.39760128e-02,
        2.95271117e-03,  

In [209]:
len(image_embeddings)

2194

In [210]:
len(image_embeddings[0])

768

## Debug embeddings 

In [211]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from transformers import BertModel, BertTokenizer

e:\DoCode\CD2\source\Source\get_hrs_rs\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [224]:
from tqdm import tqdm 

In [225]:
print("\n🤖 STEP 8 – BERT text embeddings")
xxx = pd.read_csv(os.path.join(ROOT, "items_features.csv"))

xxx["combined_text"]         = xxx["feature1"] + " " + xxx["feature2"]
xxx["cleaned_combined_text"] = xxx["combined_text"].apply(str.lower)

# ── GPU / CPU ──────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"   Device: {device}")

print("   Loading bert-base-uncased ...")
tokenizer  = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertModel.from_pretrained("bert-base-uncased")
bert_model.to(device)
bert_model.eval()

BERT_BATCH = 32   # xử lý 32 text một lúc

def get_bert_embeddings_batch(texts: list) -> np.ndarray:
    """Trả về array (N, 768) cho một batch texts."""
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        max_length=512,
        truncation=True,
        padding="max_length",
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = bert_model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).cpu().numpy()  # (N, 768)

texts = xxx["cleaned_combined_text"].tolist()
print(f"   Generating embeddings for {len(texts)} items  (batch={BERT_BATCH}) ...")

all_embeddings = []
from tqdm import tqdm as tqdm_std
for i in tqdm_std(range(0, len(texts), BERT_BATCH)):
    batch = texts[i : i + BERT_BATCH]
    all_embeddings.append(get_bert_embeddings_batch(batch))

text_embeddings = np.vstack(all_embeddings)


🤖 STEP 8 – BERT text embeddings
   Device: cpu
   Loading bert-base-uncased ...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2922.31it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   Generating embeddings for 2194 items  (batch=32) ...


  0%|          | 0/69 [00:04<?, ?it/s]


KeyboardInterrupt: 

In [219]:
xxx['combined_text'] = xxx['feature1'] + ' ' + xxx['feature2']
xxx['cleaned_combined_text'] = xxx['combined_text'].apply(preprocess_text)

# Tạo BERT embeddings cho văn bản
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4759.65it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [220]:
import torch

In [221]:
def get_bert_embeddings(text):
    inputs = tokenizer(text, return_tensors='pt', max_length=512, truncation=True, padding='max_length')
    with torch.no_grad():
        outputs = model(**inputs)
    # BERT base có kích thước embedding là 768
    embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
    return embeddings

# text_embeddings = np.vstack(
#     xxx['cleaned_combined_text'].apply(lambda x: get_bert_embeddings(x)).to_numpy()
# )

In [222]:
text_embeddings = np.vstack(
    xxx['cleaned_combined_text'].apply(lambda x: get_bert_embeddings(x)).to_numpy()
)

KeyboardInterrupt: 

In [ ]:
text_embeddings

array([[-1.32130712e-01, -4.80578125e-01,  2.61906356e-01,
         4.98509035e-04,  1.92617998e-02,  2.40128532e-01,
         1.13349840e-01,  1.76416248e-01, -1.37547314e-01,
        -3.16068172e-01, -9.78496224e-02, -1.42556384e-01,
        -1.66068539e-01,  5.95610365e-02, -1.39088720e-01,
         2.75807027e-02,  1.24800183e-01,  3.01254928e-01,
        -1.00319728e-01,  2.92990118e-01,  1.33612260e-01,
        -1.59170210e-01,  9.03117061e-02,  1.17086917e-01,
         4.02323544e-01, -5.27881086e-02, -1.23920990e-03,
         1.43134380e-02,  1.62785321e-01,  2.64486820e-01,
         1.20635331e-03, -6.07602671e-03,  1.51898682e-01,
        -1.17829554e-01,  4.28457148e-02, -3.87182057e-01,
        -1.99187919e-02, -3.18495594e-02, -4.16335613e-02,
        -9.95824933e-02, -2.50402868e-01, -6.39423504e-02,
         2.24446714e-01, -2.71887362e-01, -1.92225426e-01,
        -4.29721892e-01, -1.66247189e-01,  2.64995873e-01,
        -4.76105124e-01, -4.20513004e-02, -3.83637011e-0

In [ ]:
# Chuyển đổi về float32
text_embeddings = text_embeddings.astype(np.float32)
image_embeddings = image_embeddings.astype(np.float32)


In [ ]:
# Kiểm tra kiểu dữ liệu (dtype), kích thước (shape), và số chiều (ndim)
print("Text Embeddings:")
print("Type:", type(text_embeddings))
print("Data type (dtype):", text_embeddings.dtype)
print("Shape:", text_embeddings.shape)
print("Number of dimensions (ndim):", text_embeddings.ndim)

print("\nImage Embeddings:")
print("Type:", type(image_embeddings))
print("Data type (dtype):", image_embeddings.dtype)
print("Shape:", image_embeddings.shape)
print("Number of dimensions (ndim):", image_embeddings.ndim)


Text Embeddings:
Type: <class 'numpy.ndarray'>
Data type (dtype): float32
Shape: (1, 768)
Number of dimensions (ndim): 2

Image Embeddings:
Type: <class 'numpy.ndarray'>
Data type (dtype): float32
Shape: (8944, 768)
Number of dimensions (ndim): 2
